# 06 · Explainability (SHAP)

**Questions this notebook answers**

1. Which features drive the fraud score overall, and in which direction?
2. How does the score respond to the most important numeric features?
3. Why did the model score three specific transactions the way it did: a caught fraud,
   a missed fraud, and a false alarm?

**Method.** Exact SHAP values from the model itself (TreeSHAP via the booster's native
`pred_contrib` for tree models; coefficient × centred input for logistic regression), so no
approximation and no extra dependency. The explained model is the final pipeline **without the
calibration wrapper**, refit on all dev rows; values are in **log-odds** of that uncalibrated model.
Calibration is monotone, so it does not change the ranking the explanations describe.
One-hot columns are summed back to their source feature. Only dev rows are used.

> **Kaggle edition.** Generated from the project repository by `scripts/build_kaggle.py`. Attach the dataset [Credit Card Fraud Detection 2026](https://www.kaggle.com/datasets/uditjain13/credit-card-fraud-detection-2026) (CC0) before running. The first code cell installs the project's helper package into `/kaggle/working` (plus any results this notebook reads from earlier notebooks). Library versions on Kaggle may differ from the repo's `uv.lock`, so the last digits can differ.

In [ ]:
# Bootstrap: write the project package (and upstream artifacts) into the working
# directory. Auto-generated by scripts/build_kaggle.py.
import base64, os, sys
from pathlib import Path
WORK = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))
FILES = {'src/fraud_e2e/__init__.py': 'def hello() -> str:\n    return "Hello from fraud-e2e!"\n', 'src/fraud_e2e/config.py': '"""Project-wide constants: paths, seed, column groups.\n\nPaths resolve to the repo layout locally and to the Kaggle input mount when the\ncode runs inside a Kaggle notebook, so the same notebooks work in both places.\n"""\n\nfrom __future__ import annotations\n\nimport os\nfrom pathlib import Path\n\nSEED = 42\n\n# Repo root = two levels above this file (src/fraud_e2e/config.py).\n_REPO_ROOT = Path(__file__).resolve().parents[2]\n# Kaggle mounts datasets under /kaggle/input/<slug>/ (layout varies), outputs go to\n# /kaggle/working. Env overrides exist only so the Kaggle build can be tested locally.\n_KAGGLE_INPUT = Path(os.environ.get("KAGGLE_INPUT_DIR", "/kaggle/input"))\n_KAGGLE_WORKING = Path(os.environ.get("KAGGLE_WORKING_DIR", "/kaggle/working"))\n_RAW_NAME = "credit_card_fraud_2026.csv"\nON_KAGGLE = _KAGGLE_INPUT.is_dir() and "FRAUD_E2E_ROOT" not in os.environ\n\nif ON_KAGGLE:\n    ROOT = _KAGGLE_WORKING\n    _found = sorted(_KAGGLE_INPUT.rglob(_RAW_NAME))\n    if not _found:\n        raise FileNotFoundError(f"Attach the dataset: no {_RAW_NAME} under {_KAGGLE_INPUT}")\n    RAW_CSV = _found[0]\nelse:\n    ROOT = Path(os.environ.get("FRAUD_E2E_ROOT", _REPO_ROOT))\n    RAW_CSV = ROOT / "data" / "raw" / _RAW_NAME\n\nSPLITS_CSV = ROOT / "data" / "splits" / "splits.csv"\nREPORTS_DIR = ROOT / "reports"\nFIGURES_DIR = REPORTS_DIR / "figures"\nCONFIGS_DIR = ROOT / "configs"\n\n# SHA-256 of the canonical CSV (see data/raw/SOURCE.md). Loading fails loudly if\n# the file on disk is not byte-identical to the one the analysis was built on.\nRAW_SHA256 = "4e32b1ad4880cae452d1f2e5aa53224d38eb913c185455c00a3759df5eb3da61"\n\nID_COL = "transaction_id"\nTARGET = "is_fraud"\nAMOUNT_COL = "amount_usd"\n\nCATEGORICAL_COLS = [\n    "merchant_category",\n    "card_type",\n    "auth_method",\n    "channel",\n    "device_type",\n]\n\nBOOLEAN_COLS = [\n    "is_foreign_transaction",\n    "is_new_merchant",\n    "used_vpn",\n    "ip_country_mismatch",\n    "billing_shipping_mismatch",\n    "is_ai_generated_scam_attempt",\n]\n\n# Integer-valued columns with few distinct values (counts / clock fields).\nDISCRETE_COLS = [\n    "txn_count_last_24h",\n    "cvv_retry_count",\n    "prior_disputes",\n    "time_of_day_hour",\n    "day_of_week",\n]\n\nCONTINUOUS_COLS = [\n    "amount_usd",\n    "hours_since_last_txn",\n    "distance_from_home_km",\n    "card_age_months",\n    "customer_age",\n    "account_balance_usd",\n    "velocity_score",\n    "merchant_risk_score",\n]\n\nNUMERIC_COLS = CONTINUOUS_COLS + DISCRETE_COLS\nFEATURE_COLS = CATEGORICAL_COLS + BOOLEAN_COLS + NUMERIC_COLS\n', 'src/fraud_e2e/cost.py': '"""Business cost model for flag decisions.\n\nTwo-tier policy (primary): each scored transaction is approved or sent to\nstep-up (review / extra authentication).\n\n  TP  fraud, step-up       -> review cost\n  FP  legit, step-up       -> review cost + good-customer friction cost\n  FN  fraud, approved      -> fn_loss_fraction x amount_usd + fixed chargeback cost\n  TN  legit, approved      -> 0\n\nThree-tier policy (supplementary) adds "decline" and lets a share of fraud pass\nstep-up; see `ThreeTierParams` and `three_tier_costs`.\n\nAll values come from configs/cost.yaml. They are ILLUSTRATIVE assumptions, not\nbank data (`COST_LABEL`).\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass, replace\nfrom pathlib import Path\n\nimport numpy as np\nimport yaml\n\nfrom fraud_e2e import config as C\n\nCOST_LABEL = "Illustrative assumed costs (not bank data)"\n\n\n@dataclass(frozen=True)\nclass CostParams:\n    review_cost_per_alert: float  # USD per flagged transaction (TP and FP)\n    fp_friction_cost: float  # USD extra per wrongly flagged good customer\n    fn_loss_fraction: float  # multiplier on amount_usd lost on a missed fraud (0..2)\n    fn_fixed_cost: float  # USD per missed fraud (chargeback handling)\n    status: str = "proposed"  # proposed -> accepted (illustrative) / confirmed\n\n    def __post_init__(self):\n        for name in ("review_cost_per_alert", "fp_friction_cost", "fn_fixed_cost"):\n            if getattr(self, name) < 0:\n                raise ValueError(f"{name} must be >= 0")\n        if not 0 <= self.fn_loss_fraction <= 2:\n            raise ValueError("fn_loss_fraction must be within [0, 2]")\n\n    @property\n    def usable(self) -> bool:\n        return self.status in {"accepted", "confirmed"}\n\n    def to_dict(self) -> dict:\n        return asdict(self)\n\n    def numeric(self) -> dict:\n        d = self.to_dict()\n        d.pop("status")\n        return d\n\n\n@dataclass(frozen=True)\nclass ThreeTierParams:\n    stepup_fraud_pass_rate: float  # share of fraud that still passes step-up (0..1)\n    decline_fp_cost: float  # USD per good customer hard-declined\n    decline_tp_cost: float  # USD per fraud declined\n    status: str = "proposed"\n\n    def __post_init__(self):\n        if not 0 <= self.stepup_fraud_pass_rate <= 1:\n            raise ValueError("stepup_fraud_pass_rate must be within [0, 1]")\n        if self.decline_fp_cost < 0 or self.decline_tp_cost < 0:\n            raise ValueError("decline costs must be >= 0")\n\n    @property\n    def usable(self) -> bool:\n        return self.status in {"accepted", "confirmed"}\n\n    def to_dict(self) -> dict:\n        return asdict(self)\n\n\ndef _load_yaml(path: Path | None) -> dict:\n    path = Path(path or C.CONFIGS_DIR / "cost.yaml")\n    with open(path) as f:\n        return yaml.safe_load(f)\n\n\ndef load_cost_params(path: Path | None = None) -> CostParams:\n    raw = _load_yaml(path)\n    return CostParams(**raw["params"], status=raw.get("status", "proposed"))\n\n\ndef load_three_tier_params(path: Path | None = None) -> ThreeTierParams:\n    raw = _load_yaml(path)["three_tier"]\n    return ThreeTierParams(**raw["params"], status=raw.get("status", "proposed"))\n\n\ndef load_sensitivity_factors(path: Path | None = None) -> list[float]:\n    return list(_load_yaml(path).get("sensitivity_factors", [0.5, 2.0]))\n\n\ndef require_usable(*params) -> None:\n    for p in params:\n        if not p.usable:\n            raise PermissionError(\n                f"{type(p).__name__} status is \'{p.status}\'; it must be accepted before use"\n            )\n\n\ndef scaled(p: CostParams, name: str, factor: float) -> CostParams:\n    return replace(p, **{name: getattr(p, name) * factor})\n\n\n# --------------------------------------------------------------------------- two-tier\n\n\ndef outcome_costs(y_true, flagged, amount, p: CostParams) -> np.ndarray:\n    """Per-transaction cost in USD for a given set of flag decisions."""\n    y = np.asarray(y_true).astype(bool)\n    f = np.asarray(flagged).astype(bool)\n    a = np.asarray(amount, dtype=float)\n    cost = np.zeros(len(y))\n    cost[f] += p.review_cost_per_alert\n    cost[f & ~y] += p.fp_friction_cost\n    miss = y & ~f\n    cost[miss] += p.fn_loss_fraction * a[miss] + p.fn_fixed_cost\n    return cost\n\n\ndef total_cost(y_true, flagged, amount, p: CostParams) -> float:\n    return float(outcome_costs(y_true, flagged, amount, p).sum())\n\n\ndef reference_costs(y_true, amount, p: CostParams) -> dict:\n    """Costs of the two trivial policies every model must beat."""\n    y = np.asarray(y_true)\n    return {\n        "flag_nothing": total_cost(y, np.zeros_like(y, bool), amount, p),\n        "flag_everything": total_cost(y, np.ones_like(y, bool), amount, p),\n    }\n\n\ndef confusion_by_threshold(y_true, score, amount, thresholds) -> dict[str, np.ndarray]:\n    """TP/FP/FN/TN counts and missed-fraud USD for \'flag if score >= t\', for every t.\n\n    Cost is linear in these sufficient statistics, so they are all a front end needs to\n    recompute cost for arbitrary parameters.\n    """\n    y = np.asarray(y_true).astype(bool)\n    s = np.asarray(score, dtype=float)\n    a = np.asarray(amount, dtype=float)\n    t = np.asarray(thresholds, dtype=float)\n    order = np.argsort(s)\n    s_sorted, y_sorted, a_sorted = s[order], y[order], a[order]\n    # number of rows with score < t  (they are approved)\n    n_below = np.searchsorted(s_sorted, t, side="left")\n    cum_pos = np.r_[0, np.cumsum(y_sorted)]\n    cum_pos_amt = np.r_[0, np.cumsum(np.where(y_sorted, a_sorted, 0.0))]\n    fn = cum_pos[n_below]\n    fn_amount = cum_pos_amt[n_below]\n    tn = n_below - fn\n    tp = y.sum() - fn\n    fp = (len(y) - n_below) - tp\n    return {\n        "threshold": t,\n        "tp": tp.astype(int),\n        "fp": fp.astype(int),\n        "fn": fn.astype(int),\n        "tn": tn.astype(int),\n        "fn_amount_usd": fn_amount,\n    }\n\n\ndef cost_from_confusion(conf: dict, p: CostParams) -> np.ndarray:\n    alerts = conf["tp"] + conf["fp"]\n    return (\n        alerts * p.review_cost_per_alert\n        + conf["fp"] * p.fp_friction_cost\n        + p.fn_loss_fraction * conf["fn_amount_usd"]\n        + conf["fn"] * p.fn_fixed_cost\n    )\n\n\n# --------------------------------------------------------------------------- three-tier\n\n\ndef three_tier_costs(y_true, action, amount, p: CostParams, q: ThreeTierParams) -> np.ndarray:\n    """Per-transaction cost for actions 0 = approve, 1 = step-up, 2 = decline.\n\n    Step-up: review cost for everyone, friction for legit, and a share\n    `stepup_fraud_pass_rate` of fraud still gets through (expected FN cost).\n    Decline: `decline_fp_cost` for legit, `decline_tp_cost` for fraud.\n    """\n    y = np.asarray(y_true).astype(bool)\n    act = np.asarray(action)\n    a = np.asarray(amount, dtype=float)\n    fn_cost = p.fn_loss_fraction * a + p.fn_fixed_cost\n    cost = np.zeros(len(y))\n    approve, stepup, decline = act == 0, act == 1, act == 2\n    cost[approve & y] = fn_cost[approve & y]\n    cost[stepup] += p.review_cost_per_alert\n    cost[stepup & ~y] += p.fp_friction_cost\n    cost[stepup & y] += q.stepup_fraud_pass_rate * fn_cost[stepup & y]\n    cost[decline & ~y] = q.decline_fp_cost\n    cost[decline & y] = q.decline_tp_cost\n    return cost\n\n\ndef three_tier_actions(score, t_stepup: float, t_decline: float) -> np.ndarray:\n    s = np.asarray(score, dtype=float)\n    return np.where(s >= t_decline, 2, np.where(s >= t_stepup, 1, 0))\n', 'src/fraud_e2e/cv.py': '"""Repeated stratified CV on the dev split using the locked fold assignment."""\n\nfrom __future__ import annotations\n\nimport json\nimport time\nfrom dataclasses import dataclass, field\n\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats as ss\nfrom sklearn.metrics import average_precision_score, brier_score_loss, log_loss, roc_auc_score\n\nfrom fraud_e2e import config as C\nfrom fraud_e2e.features import feature_frame\nfrom fraud_e2e.models import fit_pipeline, normalise_spec, spec_id\n\nREPEATS = (0, 1, 2)\n\n\ndef fold_metrics(y, p) -> dict:\n    p_clip = np.clip(p, 1e-7, 1 - 1e-7)\n    return {\n        "pr_auc": average_precision_score(y, p),\n        "roc_auc": roc_auc_score(y, p),\n        "brier": brier_score_loss(y, p),\n        "log_loss": log_loss(y, p_clip, labels=[0, 1]),\n        "base_rate": float(np.mean(y)),\n    }\n\n\n@dataclass\nclass CVResult:\n    name: str\n    spec: dict\n    folds: pd.DataFrame  # one row per (repeat, fold)\n    oof: pd.DataFrame  # one row per (repeat, transaction)\n    seconds: float = 0.0\n    extra: dict = field(default_factory=dict)\n\n    def summary(self) -> dict:\n        out = {"name": self.name, "spec_id": spec_id(self.spec), "n_folds": len(self.folds)}\n        for m in ["pr_auc", "roc_auc", "brier", "log_loss"]:\n            v = self.folds[m].to_numpy()\n            out[f"{m}_mean"] = float(v.mean())\n            out[f"{m}_sd"] = float(v.std(ddof=1))\n        out["base_rate_mean"] = float(self.folds["base_rate"].mean())\n        out["seconds"] = self.seconds\n        return out\n\n\ndef run_cv(dev: pd.DataFrame, spec: dict, name: str, repeats=REPEATS) -> CVResult:\n    """Fit on 4 folds, score the 5th, for every fold of every repeat. dev must be dev-only."""\n    if "split" in dev and not dev["split"].eq("dev").all():\n        raise ValueError("run_cv received non-dev rows; the holdout must not enter CV")\n    spec = normalise_spec(spec)\n    X = feature_frame(dev, spec["drop"])\n    y = dev[C.TARGET].to_numpy()\n    rows, oofs = [], []\n    t0 = time.perf_counter()\n    for r in repeats:\n        fold_col = dev[f"fold_r{r}"].to_numpy()\n        for k in np.unique(fold_col):\n            tr, va = fold_col != k, fold_col == k\n            model = fit_pipeline(spec, X[tr], y[tr])\n            p = model.predict_proba(X[va])[:, 1]\n            rows.append(\n                {\n                    "repeat": r,\n                    "fold": int(k),\n                    "n_train": int(tr.sum()),\n                    "n_val": int(va.sum()),\n                    "n_pos_val": int(y[va].sum()),\n                    **fold_metrics(y[va], p),\n                }\n            )\n            oofs.append(\n                pd.DataFrame(\n                    {\n                        C.ID_COL: dev.loc[va, C.ID_COL].to_numpy(),\n                        "repeat": r,\n                        "fold": int(k),\n                        "y": y[va],\n                        "amount_usd": dev.loc[va, C.AMOUNT_COL].to_numpy(),\n                        "score": p,\n                    }\n                )\n            )\n    return CVResult(\n        name=name,\n        spec=spec,\n        folds=pd.DataFrame(rows),\n        oof=pd.concat(oofs, ignore_index=True),\n        seconds=time.perf_counter() - t0,\n    )\n\n\ndef corrected_ttest(\n    a: CVResult,\n    b: CVResult,\n    metric: str = "pr_auc",\n    alpha: float = 0.05,\n    lower_is_better: bool = False,\n) -> dict:\n    """Paired comparison b - a over identical folds, Nadeau & Bengio corrected resampled t-test.\n\n    Variance of the mean difference is inflated from s^2/J to (1/J + n_test/n_train) s^2\n    to account for overlapping training sets across folds and repeats.\n    """\n    key = ["repeat", "fold"]\n    m = a.folds[key + [metric, "n_train", "n_val"]].merge(\n        b.folds[key + [metric]], on=key, suffixes=("_a", "_b"), validate="one_to_one"\n    )\n    if len(m) != len(a.folds) or len(m) != len(b.folds):\n        raise ValueError("results are not on the same folds")\n    d = (m[f"{metric}_b"] - m[f"{metric}_a"]).to_numpy()\n    J = len(d)\n    ratio = float(m["n_val"].mean() / m["n_train"].mean())\n    sd = d.std(ddof=1)\n    se = np.sqrt((1 / J + ratio) * sd**2)\n    mean = float(d.mean())\n    tcrit = ss.t.ppf(1 - alpha / 2, J - 1)\n    t = mean / se if se > 0 else np.inf * np.sign(mean)\n    p = float(2 * ss.t.sf(abs(t), J - 1)) if np.isfinite(t) else 0.0\n    lo, hi = mean - tcrit * se, mean + tcrit * se\n    return {\n        "a": a.name,\n        "b": b.name,\n        "metric": metric,\n        "mean_diff_b_minus_a": mean,\n        "ci_low": float(lo),\n        "ci_high": float(hi),\n        "p_value": p,\n        "n_folds": J,\n        "b_wins_per_folds": int((d > 0).sum()),\n        "lower_is_better": lower_is_better,\n        "verdict": _verdict(lo, hi, lower_is_better),\n    }\n\n\ndef _verdict(lo: float, hi: float, lower_is_better: bool) -> str:\n    if lower_is_better:\n        lo, hi = -hi, -lo\n    if lo > 0:\n        return "b better"\n    if hi < 0:\n        return "a better"\n    return "not distinguishable"\n\n\ndef pick(\n    simpler: CVResult, complex_: CVResult, metric: str = "pr_auc", lower_is_better: bool = False\n) -> tuple[CVResult, dict]:\n    """Pre-registered rule: keep the simpler pipeline unless the complex one is clearly better."""\n    test = corrected_ttest(simpler, complex_, metric, lower_is_better=lower_is_better)\n    better = test["verdict"] == "b better"\n    test["chosen"] = complex_.name if better else simpler.name\n    return (complex_ if better else simpler), test\n\n\ndef metric_ci(res: CVResult, metric: str = "pr_auc", alpha: float = 0.05) -> dict:\n    """Mean over folds with a 95% interval using the same corrected variance as the paired test."""\n    v = res.folds[metric].to_numpy()\n    J = len(v)\n    ratio = float(res.folds["n_val"].mean() / res.folds["n_train"].mean())\n    se = np.sqrt((1 / J + ratio) * v.var(ddof=1))\n    t = ss.t.ppf(1 - alpha / 2, J - 1)\n    return {\n        "mean": float(v.mean()),\n        "sd": float(v.std(ddof=1)),\n        "ci_low": float(v.mean() - t * se),\n        "ci_high": float(v.mean() + t * se),\n    }\n\n\nCV_DIR = C.REPORTS_DIR / "cv"\n# fixed gzip header timestamp so re-runs produce byte-identical files\nGZIP = {"method": "gzip", "mtime": 0}\n\n\ndef save_result(res: CVResult, folder=CV_DIR, with_oof: bool = True) -> None:\n    """Fold metrics + spec (always committed) and OOF predictions (gzip CSV)."""\n    folder.mkdir(parents=True, exist_ok=True)\n    res.folds.to_csv(folder / f"{res.name}.folds.csv", index=False, float_format="%.6g")\n    meta = {\n        "name": res.name,\n        "spec": res.spec,\n        "spec_id": spec_id(res.spec),\n        **res.extra,\n    }\n    (folder / f"{res.name}.spec.json").write_text(json.dumps(meta, indent=2, default=str) + "\\n")\n    if with_oof:\n        res.oof.to_csv(\n            folder / f"{res.name}.oof.csv.gz",\n            index=False,\n            float_format="%.6g",\n            compression=GZIP,\n        )\n\n\ndef load_result(name: str, folder=CV_DIR) -> CVResult:\n    meta = json.loads((folder / f"{name}.spec.json").read_text())\n    folds = pd.read_csv(folder / f"{name}.folds.csv")\n    oof_path = folder / f"{name}.oof.csv.gz"\n    oof = pd.read_csv(oof_path) if oof_path.exists() else pd.DataFrame()\n    return CVResult(name=name, spec=meta["spec"], folds=folds, oof=oof)\n\n\ndef verdict_text(test: dict) -> str:\n    """Human wording of a paired-test verdict, naming the winner."""\n    if test["verdict"] == "b better":\n        return f"`{test[\'b\']}` is better"\n    if test["verdict"] == "a better":\n        return f"`{test[\'a\']}` is better"\n    return "not distinguishable"\n', 'src/fraud_e2e/data.py': '"""Loading the raw CSV and data-quality checks.\n\nEvery check returns a plain dict so results can be dumped to JSON and quoted in\ndocuments without retyping numbers by hand.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nfrom pathlib import Path\n\nimport pandas as pd\n\nfrom fraud_e2e import config as C\n\n# Allowed domain per column. Numeric ranges are *plausibility* bounds chosen\n# before looking at the data (e.g. an hour must be 0-23); they are not fitted.\nEXPECTED_LEVELS: dict[str, set[str]] = {\n    "merchant_category": {\n        "Online Retail",\n        "Groceries",\n        "Restaurants",\n        "Electronics",\n        "Fuel",\n        "Travel",\n        "Utilities",\n        "Healthcare",\n        "Gaming",\n        "Streaming",\n        "Gift Cards",\n        "Crypto Exchange",\n    },\n    "card_type": {"Visa", "Mastercard", "Amex", "RuPay", "Discover"},\n    "auth_method": {"3D Secure", "OTP", "PIN", "Biometric", "No Authentication"},\n    "channel": {"Online", "POS", "Contactless", "In-App", "ATM"},\n    "device_type": {\n        "Android Phone",\n        "iPhone",\n        "POS Terminal",\n        "Windows PC",\n        "Mac",\n        "Tablet",\n        "Smart Watch",\n        "ATM Machine",\n    },\n}\n\nRANGES: dict[str, tuple[float, float]] = {\n    "amount_usd": (0.01, 1e6),\n    "hours_since_last_txn": (0, 24 * 365),\n    "txn_count_last_24h": (0, 500),\n    "distance_from_home_km": (0, 20_100),  # half Earth\'s circumference\n    "card_age_months": (0, 600),\n    "customer_age": (18, 110),\n    "account_balance_usd": (0, 1e8),\n    "cvv_retry_count": (0, 10),\n    "velocity_score": (0, 100),\n    "time_of_day_hour": (0, 23),\n    "day_of_week": (0, 6),\n    "merchant_risk_score": (0, 100),\n    "prior_disputes": (0, 100),\n}\n\n# Cross-column combinations that would be odd for real card traffic. They are\n# reported, not removed: the dataset is used as published.\nPLAUSIBILITY_RULES: dict[str, str] = {\n    "atm_channel_on_non_atm_device": "channel == \'ATM\' and device_type != \'ATM Machine\'",\n    "atm_device_on_non_atm_channel": "device_type == \'ATM Machine\' and channel != \'ATM\'",\n    "card_present_channel_on_desktop": (\n        "channel in [\'POS\', \'Contactless\'] and device_type in [\'Windows PC\', \'Mac\']"\n    ),\n    "shipping_mismatch_at_atm": "channel == \'ATM\' and billing_shipping_mismatch",\n    "amount_exceeds_account_balance": "amount_usd > account_balance_usd",\n}\n\n\ndef sha256_of(path: Path) -> str:\n    h = hashlib.sha256()\n    with open(path, "rb") as f:\n        for chunk in iter(lambda: f.read(1 << 20), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef load_raw(path: Path | None = None, verify_hash: bool = True) -> pd.DataFrame:\n    """Read the canonical CSV with explicit dtypes."""\n    path = Path(path or C.RAW_CSV)\n    if verify_hash:\n        got = sha256_of(path)\n        if got != C.RAW_SHA256:\n            raise ValueError(f"{path} sha256 {got} != expected {C.RAW_SHA256}")\n    dtypes = {c: "category" for c in C.CATEGORICAL_COLS}\n    dtypes.update({c: "bool" for c in C.BOOLEAN_COLS})\n    df = pd.read_csv(path, dtype=dtypes)\n    df[C.TARGET] = df[C.TARGET].astype("int8")\n    return df\n\n\ndef _check(name: str, passed: bool, **detail) -> dict:\n    return {"check": name, "passed": bool(passed), **detail}\n\n\ndef quality_checks(df: pd.DataFrame) -> list[dict]:\n    """Structural checks that do not look at the label\'s relationship with features."""\n    out: list[dict] = []\n    expected_cols = [C.ID_COL, *C.FEATURE_COLS, C.TARGET]\n    out.append(\n        _check(\n            "columns_match_schema",\n            set(df.columns) == set(expected_cols),\n            missing=sorted(set(expected_cols) - set(df.columns)),\n            unexpected=sorted(set(df.columns) - set(expected_cols)),\n        )\n    )\n    out.append(_check("row_count", True, n_rows=len(df), n_cols=df.shape[1]))\n\n    n_null = df.isna().sum()\n    out.append(\n        _check(\n            "no_missing_values",\n            int(n_null.sum()) == 0,\n            missing_by_col={k: int(v) for k, v in n_null.items() if v},\n        )\n    )\n\n    ids = df[C.ID_COL]\n    out.append(_check("id_unique", ids.is_unique, n_unique=int(ids.nunique())))\n    out.append(\n        _check(\n            "id_contiguous_1_to_n",\n            bool((ids.sort_values().to_numpy() == range(1, len(df) + 1)).all()),\n        )\n    )\n\n    n_dup = int(df.drop(columns=C.ID_COL).duplicated().sum())\n    out.append(_check("no_duplicate_rows_excluding_id", n_dup == 0, n_duplicates=n_dup))\n    n_dup_x = int(df[C.FEATURE_COLS].duplicated().sum())\n    out.append(_check("no_duplicate_feature_vectors", n_dup_x == 0, n_duplicates=n_dup_x))\n\n    labels = sorted(df[C.TARGET].unique().tolist())\n    out.append(\n        _check(\n            "target_is_binary",\n            labels == [0, 1],\n            values=labels,\n            n_positive=int(df[C.TARGET].sum()),\n            positive_rate=float(df[C.TARGET].mean()),\n        )\n    )\n\n    for col, allowed in EXPECTED_LEVELS.items():\n        seen = set(df[col].astype(str).unique())\n        out.append(\n            _check(\n                f"levels_{col}",\n                seen <= allowed,\n                unexpected=sorted(seen - allowed),\n                n_levels=len(seen),\n            )\n        )\n\n    for col, (lo, hi) in RANGES.items():\n        s = df[col]\n        bad = int(((s < lo) | (s > hi)).sum())\n        out.append(\n            _check(\n                f"range_{col}",\n                bad == 0,\n                n_out_of_range=bad,\n                observed_min=float(s.min()),\n                observed_max=float(s.max()),\n                allowed=[lo, hi],\n            )\n        )\n    return out\n\n\ndef plausibility_checks(df: pd.DataFrame) -> list[dict]:\n    """Count rows matching each implausible cross-column combination."""\n    out = []\n    for name, expr in PLAUSIBILITY_RULES.items():\n        n = int(df.eval(expr).sum())\n        out.append({"rule": name, "expression": expr, "n_rows": n, "share_of_rows": n / len(df)})\n    return out\n', 'src/fraud_e2e/diagnostics.py': '"""Diagnostics that explain modelling results (not used for selection)."""\n\nfrom __future__ import annotations\n\nimport pandas as pd\n\nfrom fraud_e2e import config as C\nfrom fraud_e2e.features import feature_frame\nfrom fraud_e2e.models import build_pipeline\n\n\ndef smote_synthetic_vs_real(dev: pd.DataFrame, spec: dict, repeat: int = 0, fold: int = 0):\n    """Red-flag prevalence in real vs SMOTENC-generated fraud, for one training fold."""\n    tr = dev[dev[f"fold_r{repeat}"] != fold]\n    X, y = feature_frame(tr), tr[C.TARGET].to_numpy()\n    sampler = build_pipeline({**spec, "imbalance": "smote"}).named_steps["sampler"]\n    Xs, _ = sampler.fit_resample(X, y)\n    synthetic = Xs.iloc[len(X) :]\n    real = X[y == 1]\n    cols = C.BOOLEAN_COLS + ["cvv_retry_count", "merchant_risk_score", "velocity_score"]\n    out = pd.DataFrame(\n        {\n            "feature": cols,\n            "real_fraud_mean": [float(real[c].mean()) for c in cols],\n            "synthetic_fraud_mean": [float(synthetic[c].astype(float).mean()) for c in cols],\n            "legit_mean": [float(X.loc[y == 0, c].mean()) for c in cols],\n        }\n    )\n    out["n_real_fraud"], out["n_synthetic"] = len(real), len(synthetic)\n    return out\n', 'src/fraud_e2e/evaluate.py': '"""One-time holdout evaluation, guarded by a lock file.\n\nThe lock records exactly what was evaluated (pipeline spec id + thresholds).\nRe-running with the *same* frozen choices just reproduces the same numbers; any\nattempt to evaluate a *different* pipeline or threshold on the holdout raises,\nso the holdout cannot be used for selection by accident.\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\nfrom fraud_e2e import config as C\nfrom fraud_e2e.cost import CostParams, outcome_costs, three_tier_actions, three_tier_costs\nfrom fraud_e2e.features import feature_frame\nfrom fraud_e2e.models import fit_pipeline, normalise_spec, spec_id\n\nLOCK_PATH = C.ROOT / "notes" / "holdout_lock.json"\n\n\ndef fit_final(dev: pd.DataFrame, spec: dict):\n    """Refit the chosen pipeline on all dev rows."""\n    if "split" in dev and not dev["split"].eq("dev").all():\n        raise ValueError("final model must be fit on dev rows only")\n    spec = normalise_spec(spec)\n    return fit_pipeline(spec, feature_frame(dev, spec["drop"]), dev[C.TARGET].to_numpy())\n\n\ndef _check_lock(frozen: dict, lock_path: Path, allow_create: bool) -> None:\n    if lock_path.exists():\n        locked = json.loads(lock_path.read_text())\n        if locked != frozen:\n            raise PermissionError(\n                f"Holdout already evaluated with different choices ({lock_path}). "\n                "Refusing to evaluate a new pipeline/threshold on the holdout."\n            )\n    elif allow_create:\n        lock_path.parent.mkdir(parents=True, exist_ok=True)\n        lock_path.write_text(json.dumps(frozen, indent=2) + "\\n")\n    else:\n        raise PermissionError("No holdout lock yet; pass allow_create=True for the one-time run")\n\n\ndef _metrics(y, s, amount, t_flag, p: CostParams, three_tier=None) -> dict:\n    flag = s >= t_flag\n    tp = int((flag & (y == 1)).sum())\n    fp = int((flag & (y == 0)).sum())\n    fn = int((~flag & (y == 1)).sum())\n    out = {\n        "pr_auc": average_precision_score(y, s),\n        "roc_auc": roc_auc_score(y, s),\n        "base_rate": float(y.mean()),\n        "recall": tp / (tp + fn) if tp + fn else np.nan,\n        "precision": tp / (tp + fp) if tp + fp else np.nan,\n        "alert_rate": float(flag.mean()),\n        "cost_per_1000_txn": float(outcome_costs(y, flag, amount, p).sum() / len(y) * 1000),\n        "flag_nothing_cost_per_1000_txn": float(\n            outcome_costs(y, np.zeros_like(flag), amount, p).sum() / len(y) * 1000\n        ),\n    }\n    if three_tier is not None:\n        q, t1, t2 = three_tier\n        act = three_tier_actions(s, t1, t2)\n        out["three_tier_cost_per_1000_txn"] = float(\n            three_tier_costs(y, act, amount, p, q).sum() / len(y) * 1000\n        )\n        out["three_tier_decline_rate"] = float((act == 2).mean())\n        out["three_tier_stepup_rate"] = float((act == 1).mean())\n    return out\n\n\ndef bootstrap(fn, y, s, amount, n_boot=2000, seed=C.SEED) -> dict:\n    """Stratified bootstrap (fraud and legit resampled separately) percentile 95% CIs.\n\n    `fn(y, s, amount) -> dict of scalars` is evaluated on the original sample and on each\n    resample.\n    """\n    rng = np.random.default_rng(seed)\n    y, s, amount = np.asarray(y), np.asarray(s, float), np.asarray(amount, float)\n    pos, neg = np.flatnonzero(y == 1), np.flatnonzero(y == 0)\n    point = fn(y, s, amount)\n    reps = []\n    for _ in range(n_boot):\n        idx = np.r_[rng.choice(pos, len(pos)), rng.choice(neg, len(neg))]\n        reps.append(fn(y[idx], s[idx], amount[idx]))\n    reps = pd.DataFrame(reps)\n    return {\n        k: {\n            "value": float(v),\n            "ci_low": float(reps[k].quantile(0.025)),\n            "ci_high": float(reps[k].quantile(0.975)),\n        }\n        for k, v in point.items()\n    }\n\n\ndef bootstrap_metrics(y, s, amount, t_flag, p, three_tier=None, n_boot=2000, seed=C.SEED):\n    """Pre-registered holdout metrics with stratified bootstrap CIs."""\n    return bootstrap(\n        lambda yy, ss_, aa: _metrics(yy, ss_, aa, t_flag, p, three_tier), y, s, amount, n_boot, seed\n    )\n\n\ndef evaluate_holdout(\n    df_all: pd.DataFrame,\n    spec: dict,\n    t_flag: float,\n    p: CostParams,\n    three_tier=None,\n    lock_path: Path = LOCK_PATH,\n    allow_create: bool = False,\n    n_boot: int = 2000,\n) -> dict:\n    """Fit on dev, score holdout once, bootstrap CIs. three_tier = (ThreeTierParams, t1, t2)."""\n    spec = normalise_spec(spec)\n    frozen = {\n        "spec_id": spec_id(spec),\n        "spec": spec,\n        "t_flag": round(float(t_flag), 10),\n        "cost_params": p.to_dict(),\n        "three_tier": None\n        if three_tier is None\n        else {\n            "params": three_tier[0].to_dict(),\n            "t_stepup": round(float(three_tier[1]), 10),\n            "t_decline": round(float(three_tier[2]), 10),\n        },\n    }\n    _check_lock(frozen, lock_path, allow_create)\n    dev = df_all[df_all["split"] == "dev"]\n    hold = df_all[df_all["split"] == "holdout"]\n    model = fit_final(dev, spec)\n    s = model.predict_proba(feature_frame(hold, spec["drop"]))[:, 1]\n    y = hold[C.TARGET].to_numpy()\n    res = bootstrap_metrics(y, s, hold[C.AMOUNT_COL].to_numpy(), t_flag, p, three_tier, n_boot)\n    return {\n        "frozen": frozen,\n        "n_rows": len(hold),\n        "n_fraud": int(y.sum()),\n        "metrics": res,\n        "_scores": pd.DataFrame({C.ID_COL: hold[C.ID_COL].to_numpy(), "y": y, "score": s}),\n    }\n', 'src/fraud_e2e/explain.py': '"""SHAP values without the `shap` package.\n\nTree models: exact TreeSHAP from the booster itself (LightGBM `pred_contrib`,\nXGBoost `pred_contribs`). Logistic regression: exact linear SHAP,\ncoef x (x - mean(x)) on the standardised design matrix. Values are in log-odds\nof the *uncalibrated* model. One-hot columns are summed back to their source\nfeature so a categorical feature gets one value per transaction.\n"""\n\nfrom __future__ import annotations\n\nimport pandas as pd\nfrom sklearn.calibration import CalibratedClassifierCV\n\nfrom fraud_e2e.features import feature_frame, source_feature\n\n\ndef _base_pipeline(model):\n    """The fitted pipeline to explain. For a calibrated ensemble use the first member."""\n    if isinstance(model, CalibratedClassifierCV):\n        return model.calibrated_classifiers_[0].estimator\n    return model\n\n\ndef shap_values(model, df: pd.DataFrame, drop=()) -> tuple[pd.DataFrame, float]:\n    """Per-row SHAP per source feature (log-odds), and the expected value (base)."""\n    pipe = _base_pipeline(model)\n    prep, est = pipe.named_steps["prep"], pipe.named_steps["model"]\n    Xt = prep.transform(feature_frame(df, drop))\n    names = prep.get_feature_names_out()\n    kind = type(est).__name__\n    if kind == "LGBMClassifier":\n        contrib = est.booster_.predict(Xt, pred_contrib=True)\n        phi, base = contrib[:, :-1], float(contrib[0, -1])\n    elif kind == "XGBClassifier":\n        import xgboost as xgb\n\n        contrib = est.get_booster().predict(xgb.DMatrix(Xt, feature_names=None), pred_contribs=True)\n        phi, base = contrib[:, :-1], float(contrib[0, -1])\n    elif kind == "LogisticRegression":\n        mu = Xt.mean(axis=0)\n        phi = (Xt - mu) * est.coef_[0]\n        base = float(est.intercept_[0] + mu @ est.coef_[0])\n    else:\n        raise TypeError(f"no SHAP implementation for {kind}")\n    by_col = pd.DataFrame(phi, columns=names, index=df.index)\n    groups = [source_feature(n, drop) for n in names]\n    return by_col.T.groupby(groups, sort=False).sum().T, base\n\n\ndef global_importance(phi: pd.DataFrame) -> pd.DataFrame:\n    imp = phi.abs().mean().sort_values(ascending=False)\n    return imp.rename("mean_abs_shap").reset_index().rename(columns={"index": "feature"})\n', 'src/fraud_e2e/features.py': '"""Feature matrix and preprocessing (fit inside each training fold by the pipeline)."""\n\nfrom __future__ import annotations\n\nfrom collections.abc import Sequence\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.compose import ColumnTransformer\nfrom sklearn.pipeline import make_pipeline\nfrom sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler\n\nfrom fraud_e2e import config as C\n\nLOG_COLS = ["amount_usd", "account_balance_usd"]  # heavy right tails (P1)\n\n\ndef feature_list(drop: Sequence[str] = ()) -> list[str]:\n    unknown = set(drop) - set(C.FEATURE_COLS)\n    if unknown:\n        raise ValueError(f"Unknown features to drop: {sorted(unknown)}")\n    return [c for c in C.FEATURE_COLS if c not in set(drop)]\n\n\ndef feature_frame(df: pd.DataFrame, drop: Sequence[str] = ()) -> pd.DataFrame:\n    """Model input: feature columns only (never the ID or target), booleans as 0/1."""\n    cols = feature_list(drop)\n    X = df[cols].copy()\n    for c in C.BOOLEAN_COLS:\n        if c in X:\n            X[c] = X[c].astype(np.int8)\n    return X\n\n\ndef make_preprocessor(kind: str, drop: Sequence[str] = ()) -> ColumnTransformer:\n    """\'linear\': one-hot + log1p(heavy tails) + standardise; \'tree\': one-hot, raw numerics."""\n    cols = feature_list(drop)\n    cats = [c for c in C.CATEGORICAL_COLS if c in cols]\n    bools = [c for c in C.BOOLEAN_COLS if c in cols]\n    nums = [c for c in C.NUMERIC_COLS if c in cols]\n    onehot = OneHotEncoder(handle_unknown="ignore", sparse_output=False, dtype=np.float64)\n    if kind == "linear":\n        logs = [c for c in nums if c in LOG_COLS]\n        plain = [c for c in nums if c not in LOG_COLS]\n        transformers = [\n            ("cat", onehot, cats),\n            ("bool", "passthrough", bools),\n            (\n                "log",\n                make_pipeline(\n                    FunctionTransformer(np.log1p, feature_names_out="one-to-one"), StandardScaler()\n                ),\n                logs,\n            ),\n            ("num", StandardScaler(), plain),\n        ]\n    elif kind == "tree":\n        transformers = [\n            ("cat", onehot, cats),\n            ("bool", "passthrough", bools),\n            ("num", "passthrough", nums),\n        ]\n    else:\n        raise ValueError(f"unknown preprocessor kind {kind!r}")\n    return ColumnTransformer(transformers, verbose_feature_names_out=True)\n\n\ndef source_feature(transformed_name: str, drop: Sequence[str] = ()) -> str:\n    """Map an output column (e.g. \'cat__auth_method_PIN\') back to its input feature."""\n    _, _, rest = transformed_name.partition("__")\n    if rest in C.FEATURE_COLS:\n        return rest\n    for c in sorted(feature_list(drop), key=len, reverse=True):\n        if rest.startswith(c + "_"):\n            return c\n    raise KeyError(transformed_name)\n', 'src/fraud_e2e/models.py': '"""Model pipelines described by a plain-dict spec, so every experiment is loggable.\n\nspec = {\n    "model": "logreg" | "lgbm" | "xgb",\n    "params": {...},               # overrides of the defaults below\n    "imbalance": "none" | "class_weight" | "undersample" | "smote",\n    "drop": [...],                 # features to leave out (ablation)\n    "calibration": None | "sigmoid" | "isotonic",\n}\nSamplers sit inside an imblearn Pipeline, so they only ever see training rows.\n"""\n\nfrom __future__ import annotations\n\nimport copy\nimport hashlib\nimport json\n\nfrom imblearn.over_sampling import SMOTENC\nfrom imblearn.pipeline import Pipeline\nfrom imblearn.under_sampling import RandomUnderSampler\nfrom lightgbm import LGBMClassifier\nfrom sklearn.calibration import CalibratedClassifierCV\nfrom sklearn.linear_model import LogisticRegression\nfrom xgboost import XGBClassifier\n\nfrom fraud_e2e import config as C\nfrom fraud_e2e.features import feature_list, make_preprocessor\n\nN_JOBS = 4  # keep the Mac responsive for other projects\n\nDEFAULTS: dict[str, dict] = {\n    "logreg": {"C": 1.0, "max_iter": 2000},\n    "lgbm": {\n        "n_estimators": 300,\n        "learning_rate": 0.05,\n        "num_leaves": 15,\n        "min_child_samples": 20,\n        "subsample": 0.8,\n        "subsample_freq": 1,\n        "colsample_bytree": 0.8,\n        "reg_lambda": 0.0,\n    },\n    "xgb": {\n        "n_estimators": 300,\n        "learning_rate": 0.05,\n        "max_depth": 4,\n        "min_child_weight": 20 * 0.0169 * (1 - 0.0169),  # ~20 rows\' worth of hessian\n        "subsample": 0.8,\n        "colsample_bytree": 0.8,\n        "reg_lambda": 1.0,\n    },\n}\n\nUNDERSAMPLE_RATIO = 0.1  # minority / majority after undersampling (legit 10:1)\nSMOTE_RATIO = 0.1  # minority / majority after SMOTENC\nSMOTE_K = 5\n\n\ndef spec_id(spec: dict) -> str:\n    """Short stable hash of a spec (for logs and the holdout lock)."""\n    blob = json.dumps(spec, sort_keys=True, default=str).encode()\n    return hashlib.sha256(blob).hexdigest()[:12]\n\n\ndef normalise_spec(spec: dict) -> dict:\n    s = {"params": {}, "imbalance": "none", "drop": [], "calibration": None, **copy.deepcopy(spec)}\n    s["drop"] = sorted(s["drop"])\n    return s\n\n\ndef _estimator(spec: dict):\n    name = spec["model"]\n    params = {**DEFAULTS[name], **spec["params"]}\n    weighted = spec["imbalance"] == "class_weight"\n    if name == "logreg":\n        return LogisticRegression(\n            **params, class_weight="balanced" if weighted else None, random_state=C.SEED\n        )\n    if name == "lgbm":\n        return LGBMClassifier(\n            **params,\n            class_weight="balanced" if weighted else None,\n            random_state=C.SEED,\n            n_jobs=N_JOBS,\n            deterministic=True,\n            force_row_wise=True,\n            verbose=-1,\n        )\n    if name == "xgb":\n        # scale_pos_weight is set from the training fold by fit_pipeline when weighted\n        return XGBClassifier(\n            **params,\n            tree_method="hist",\n            random_state=C.SEED,\n            n_jobs=N_JOBS,\n            eval_metric="logloss",\n        )\n    raise ValueError(f"unknown model {name!r}")\n\n\ndef build_pipeline(spec: dict):\n    spec = normalise_spec(spec)\n    kind = "linear" if spec["model"] == "logreg" else "tree"\n    steps = []\n    if spec["imbalance"] == "undersample":\n        steps.append(\n            (\n                "sampler",\n                RandomUnderSampler(sampling_strategy=UNDERSAMPLE_RATIO, random_state=C.SEED),\n            )\n        )\n    elif spec["imbalance"] == "smote":\n        cols = feature_list(spec["drop"])\n        cat_like = [c for c in cols if c in C.CATEGORICAL_COLS + C.BOOLEAN_COLS]\n        steps.append(\n            (\n                "sampler",\n                SMOTENC(\n                    categorical_features=cat_like,\n                    sampling_strategy=SMOTE_RATIO,\n                    k_neighbors=SMOTE_K,\n                    random_state=C.SEED,\n                ),\n            )\n        )\n    elif spec["imbalance"] not in {"none", "class_weight"}:\n        raise ValueError(f"unknown imbalance strategy {spec[\'imbalance\']!r}")\n    steps += [("prep", make_preprocessor(kind, spec["drop"])), ("model", _estimator(spec))]\n    pipe = Pipeline(steps)\n    if spec["calibration"]:\n        return CalibratedClassifierCV(pipe, method=spec["calibration"], cv=5, ensemble=True)\n    return pipe\n\n\ndef fit_pipeline(spec: dict, X, y):\n    """Build and fit; handles XGBoost\'s class weight, which needs the fold\'s class ratio."""\n    spec = normalise_spec(spec)\n    pipe = build_pipeline(spec)\n    if spec["model"] == "xgb" and spec["imbalance"] == "class_weight":\n        ratio = float((y == 0).sum() / (y == 1).sum())\n        key = (\n            "estimator__model__scale_pos_weight"\n            if spec["calibration"]\n            else "model__scale_pos_weight"\n        )\n        pipe.set_params(**{key: ratio})\n    return pipe.fit(X, y)\n', 'src/fraud_e2e/plots.py': '"""Matplotlib helpers with one consistent, readable style.\n\nColour roles (fixed, never cycled): legit = blue, fraud = orange, reference\nlines = neutral grey. Every plot sets axis labels with units.\n"""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\n\nfrom fraud_e2e import config as C\n\nLEGIT = "#2a78d6"\nFRAUD = "#eb6834"\nNEUTRAL = "#8a8985"\nINK = "#0b0b0b"\nINK_2 = "#52514e"\nGRID = "#e4e3df"\n\n\ndef set_style() -> None:\n    plt.rcParams.update(\n        {\n            "figure.dpi": 110,\n            "savefig.dpi": 150,\n            "savefig.bbox": "tight",\n            "figure.facecolor": "#fcfcfb",\n            "axes.facecolor": "#fcfcfb",\n            "axes.edgecolor": GRID,\n            "axes.labelcolor": INK_2,\n            "axes.titlecolor": INK,\n            "axes.titlesize": 12,\n            "axes.titleweight": "bold",\n            "axes.labelsize": 10,\n            "axes.spines.top": False,\n            "axes.spines.right": False,\n            "axes.grid": True,\n            "grid.color": GRID,\n            "grid.linewidth": 0.8,\n            "xtick.color": INK_2,\n            "ytick.color": INK_2,\n            "legend.frameon": False,\n            "font.size": 10,\n        }\n    )\n\n\ndef save(fig, name: str, folder: Path = C.FIGURES_DIR) -> Path:\n    folder.mkdir(parents=True, exist_ok=True)\n    path = folder / f"{name}.png"\n    fig.savefig(path)\n    return path\n\n\ndef rate_bars(table: pd.DataFrame, base_rate: float, ax=None, title: str | None = None):\n    """Horizontal bars of fraud rate per level with 95% Wilson CI whiskers."""\n    t = table.sort_values("fraud_rate")\n    ax = ax or plt.subplots(figsize=(6, 0.35 * len(t) + 1.2))[1]\n    y = np.arange(len(t))\n    ax.barh(y, t["fraud_rate"] * 100, color=FRAUD, height=0.6, zorder=2)\n    err = [(t["fraud_rate"] - t["ci_low"]) * 100, (t["ci_high"] - t["fraud_rate"]) * 100]\n    ax.errorbar(\n        t["fraud_rate"] * 100,\n        y,\n        xerr=err,\n        fmt="none",\n        ecolor=INK_2,\n        elinewidth=1,\n        capsize=2,\n        zorder=3,\n    )\n    ax.axvline(base_rate * 100, color=NEUTRAL, ls="--", lw=1, zorder=1)\n    ax.set_yticks(y, [f"{lvl}  (n={n:,})" for lvl, n in zip(t["level"], t["n"])])\n    ax.set_xlabel("Fraud rate (%) — bars: 95% Wilson CI, dashed: overall rate")\n    ax.grid(axis="y", visible=False)\n    ax.set_title(title or t["feature"].iloc[0])\n    return ax\n\n\ndef binned_rate_line(\n    table: pd.DataFrame, base_rate: float, xlabel: str, ax=None, logx: bool = False\n):\n    """Fraud rate across quantile bins of a numeric feature, CI as a band."""\n    ax = ax or plt.subplots(figsize=(5, 3.2))[1]\n    x = table["bin_mid"].astype(float)\n    ax.fill_between(x, table["ci_low"] * 100, table["ci_high"] * 100, color=FRAUD, alpha=0.18, lw=0)\n    ax.plot(x, table["fraud_rate"] * 100, color=FRAUD, lw=2, marker="o", ms=4)\n    ax.axhline(base_rate * 100, color=NEUTRAL, ls="--", lw=1)\n    if logx:\n        ax.set_xscale("log")\n    ax.set_xlabel(xlabel)\n    ax.set_ylabel("Fraud rate (%)")\n    ax.set_ylim(bottom=0)\n    return ax\n\n\ndef class_hist(\n    df: pd.DataFrame,\n    col: str,\n    xlabel: str,\n    ax=None,\n    bins: int = 40,\n    logx: bool = False,\n    target: str = C.TARGET,\n):\n    """Overlaid histograms for legit vs fraud, each class normalised to 100%."""\n    ax = ax or plt.subplots(figsize=(5, 3.2))[1]\n    x = df[col].astype(float)\n    if logx:\n        edges = np.geomspace(max(x.min(), 1e-2), x.max(), bins + 1)\n    else:\n        edges = np.histogram_bin_edges(x, bins=bins)\n    for label, colour, name in [(0, LEGIT, "Legit"), (1, FRAUD, "Fraud")]:\n        xs = x[df[target] == label]\n        ax.hist(\n            xs,\n            bins=edges,\n            weights=np.full(len(xs), 100 / len(xs)),\n            histtype="step",\n            lw=2,\n            color=colour,\n            label=f"{name} (n={len(xs):,})",\n        )\n    if logx:\n        ax.set_xscale("log")\n    ax.set_xlabel(xlabel)\n    ax.set_ylabel("Share of class in bin (%)")\n    ax.legend()\n    return ax\n\n\ndef auc_forest(table: pd.DataFrame, ax=None):\n    """Univariate ROC AUC per feature with bootstrap CI; 0.5 = no signal."""\n    t = table.sort_values("separation")\n    ax = ax or plt.subplots(figsize=(6, 0.33 * len(t) + 1.2))[1]\n    y = np.arange(len(t))\n    colours = [FRAUD if a > 0.5 else LEGIT for a in t["auc"]]\n    ax.hlines(y, t["auc_ci_low"], t["auc_ci_high"], color=INK_2, lw=1.5)\n    ax.scatter(t["auc"], y, color=colours, s=40, zorder=3, edgecolor="#fcfcfb", lw=1.5)\n    ax.axvline(0.5, color=NEUTRAL, ls="--", lw=1)\n    ax.set_yticks(y, t["feature"])\n    ax.set_xlabel("Univariate ROC AUC (95% bootstrap CI); >0.5 orange = higher value → more fraud")\n    ax.grid(axis="y", visible=False)\n    return ax\n\n\nSERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]\n\n\ndef fold_dots(results: list, metric: str = "pr_auc", ax=None, xlabel: str | None = None):\n    """Per-fold metric for each pipeline (dots), with mean and corrected 95% CI (bar)."""\n    from fraud_e2e.cv import metric_ci\n\n    ax = ax or plt.subplots(figsize=(7, 0.55 * len(results) + 1.3))[1]\n    rng = np.random.default_rng(0)\n    for i, res in enumerate(results):\n        v = res.folds[metric].to_numpy()\n        ci = metric_ci(res, metric)\n        y = len(results) - 1 - i\n        ax.scatter(\n            v,\n            y + rng.uniform(-0.12, 0.12, len(v)),\n            s=14,\n            color=SERIES[0],\n            alpha=0.45,\n            lw=0,\n            zorder=2,\n        )\n        ax.hlines(y, ci["ci_low"], ci["ci_high"], color=INK, lw=2, zorder=3)\n        ax.scatter([ci["mean"]], [y], s=60, color=FRAUD, zorder=4, edgecolor="#fcfcfb", lw=1.5)\n    ax.set_yticks(range(len(results)), [r.name for r in results][::-1])\n    ax.set_xlabel(xlabel or f"{metric} per validation fold (dots); mean ± corrected 95% CI")\n    ax.grid(axis="y", visible=False)\n    return ax\n\n\ndef pr_curves(results: list, ax=None, repeat: int = 0):\n    """Precision-recall curves of OOF predictions (one repeat), with the no-skill line."""\n    from sklearn.metrics import precision_recall_curve\n\n    ax = ax or plt.subplots(figsize=(5.5, 4))[1]\n    base = None\n    for i, res in enumerate(results):\n        o = res.oof[res.oof["repeat"] == repeat]\n        prec, rec, _ = precision_recall_curve(o["y"], o["score"])\n        ax.plot(rec * 100, prec * 100, color=SERIES[i % len(SERIES)], lw=2, label=res.name)\n        base = o["y"].mean()\n    ax.axhline(base * 100, color=NEUTRAL, ls="--", lw=1, label=f"No skill ({100 * base:.2f}%)")\n    ax.set_xlabel("Recall (% of fraud caught)")\n    ax.set_ylabel("Precision (% of alerts that are fraud)")\n    ax.set_xlim(0, 100)\n    ax.set_ylim(0, 100)\n    ax.legend(loc="upper right")\n    return ax\n', 'src/fraud_e2e/reporting.py': '"""Write analysis results as committed artifacts (JSON / CSV).\n\nDocuments and the website quote numbers only from these files, so every\nartifact carries the data hash and seed it was produced with.\n"""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nfrom fraud_e2e import config as C\n\n\ndef _default(o):\n    if isinstance(o, (np.integer,)):\n        return int(o)\n    if isinstance(o, (np.floating,)):\n        return float(o)\n    if isinstance(o, (np.bool_,)):\n        return bool(o)\n    if isinstance(o, np.ndarray):\n        return o.tolist()\n    if isinstance(o, pd.DataFrame):\n        return o.to_dict(orient="records")\n    raise TypeError(f"Not JSON serialisable: {type(o)}")\n\n\ndef write_json(payload: dict, name: str, folder: Path = C.REPORTS_DIR) -> Path:\n    folder.mkdir(parents=True, exist_ok=True)\n    path = folder / f"{name}.json"\n    body = {"_meta": {"data_sha256": C.RAW_SHA256, "seed": C.SEED}, **payload}\n    path.write_text(json.dumps(body, indent=2, default=_default, sort_keys=False) + "\\n")\n    return path\n\n\ndef write_csv(df: pd.DataFrame, name: str, folder: Path = C.REPORTS_DIR) -> Path:\n    folder.mkdir(parents=True, exist_ok=True)\n    path = folder / f"{name}.csv"\n    df.to_csv(path, index=False, float_format="%.6g")\n    return path\n\n\ndef read_json(name: str, folder: Path = C.REPORTS_DIR) -> dict:\n    return json.loads((folder / f"{name}.json").read_text())\n', 'src/fraud_e2e/splits.py': '"""Locked data splits.\n\nDesign (fixed before any label-based EDA, see notes/00_preregistration.md):\n  * 20% stratified holdout, touched once at the very end of modelling.\n  * The remaining 80% ("dev") gets repeated stratified 5-fold CV assignments\n    (3 repeats, different shuffles) so fold-to-fold variance can be reported.\nThe assignments are written to data/splits/splits.csv and committed; every\nlater stage reads that file instead of re-splitting.\n"""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedKFold, train_test_split\n\nfrom fraud_e2e import config as C\n\nHOLDOUT_FRACTION = 0.20\nN_FOLDS = 5\nN_REPEATS = 3\n\n\ndef make_splits(df: pd.DataFrame, seed: int = C.SEED) -> pd.DataFrame:\n    """Return one row per transaction: split (\'dev\'/\'holdout\') and fold_r{k} for dev rows."""\n    ids = df[C.ID_COL].to_numpy()\n    y = df[C.TARGET].to_numpy()\n    dev_ids, _ = train_test_split(ids, test_size=HOLDOUT_FRACTION, stratify=y, random_state=seed)\n    out = pd.DataFrame({C.ID_COL: ids, "split": "holdout"})\n    out.loc[out[C.ID_COL].isin(dev_ids), "split"] = "dev"\n\n    dev_mask = out["split"].eq("dev").to_numpy()\n    y_dev = y[dev_mask]\n    for r in range(N_REPEATS):\n        fold = np.full(len(out), -1, dtype=int)\n        skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed + r)\n        dev_fold = np.empty(dev_mask.sum(), dtype=int)\n        for k, (_, val_idx) in enumerate(skf.split(np.zeros(len(y_dev)), y_dev)):\n            dev_fold[val_idx] = k\n        fold[dev_mask] = dev_fold\n        out[f"fold_r{r}"] = fold\n    return out\n\n\ndef save_splits(splits: pd.DataFrame, path: Path = C.SPLITS_CSV) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    splits.to_csv(path, index=False)\n\n\ndef load_splits(path: Path = C.SPLITS_CSV) -> pd.DataFrame:\n    return pd.read_csv(path)\n\n\ndef attach_splits(df: pd.DataFrame, splits: pd.DataFrame | None = None) -> pd.DataFrame:\n    splits = load_splits() if splits is None else splits\n    merged = df.merge(splits, on=C.ID_COL, how="left", validate="one_to_one")\n    if merged["split"].isna().any():\n        raise ValueError("Some transactions have no split assignment")\n    return merged\n\n\ndef dev_only(df: pd.DataFrame) -> pd.DataFrame:\n    """The only slice that label-based EDA and model selection may look at."""\n    if "split" not in df.columns:\n        df = attach_splits(df)\n    return df[df["split"].eq("dev")].reset_index(drop=True)\n', 'src/fraud_e2e/stats.py': '"""Small statistics helpers for a rare-event target.\n\nWith only a few hundred positives, every rate and score here is reported with\nan interval. Intervals are Wilson (for proportions) or stratified bootstrap\npercentile intervals (for AUC), both seeded for reproducibility.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\nfrom scipy import stats as ss\n\nfrom fraud_e2e import config as C\n\n\ndef wilson_ci(k: np.ndarray | int, n: np.ndarray | int, alpha: float = 0.05):\n    """Wilson score interval for k successes out of n. Vectorised."""\n    k = np.asarray(k, dtype=float)\n    n = np.asarray(n, dtype=float)\n    z = ss.norm.ppf(1 - alpha / 2)\n    p = np.divide(k, n, out=np.zeros_like(k), where=n > 0)\n    denom = 1 + z**2 / n\n    centre = (p + z**2 / (2 * n)) / denom\n    half = z * np.sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom\n    lo = np.where(k == 0, 0.0, np.clip(centre - half, 0, 1))  # exact bounds at the edges\n    hi = np.where(k == n, 1.0, np.clip(centre + half, 0, 1))\n    return lo, hi\n\n\ndef rate_table(df: pd.DataFrame, col: str, target: str = C.TARGET) -> pd.DataFrame:\n    """Fraud rate per level of `col` with 95% Wilson CI and lift vs the overall rate."""\n    g = df.groupby(col, observed=True)[target].agg(n="size", n_fraud="sum").reset_index()\n    g["fraud_rate"] = g["n_fraud"] / g["n"]\n    g["ci_low"], g["ci_high"] = wilson_ci(g["n_fraud"].to_numpy(), g["n"].to_numpy())\n    base = df[target].mean()\n    g["lift"] = g["fraud_rate"] / base\n    g.insert(0, "feature", col)\n    g = g.rename(columns={col: "level"})\n    g["level"] = g["level"].astype(str)\n    return g.sort_values("fraud_rate", ascending=False).reset_index(drop=True)\n\n\ndef chi2_association(df: pd.DataFrame, col: str, target: str = C.TARGET) -> dict:\n    """Chi-square test of independence + Cramér\'s V between a discrete column and the target."""\n    table = pd.crosstab(df[col], df[target])\n    chi2, p, dof, _ = ss.chi2_contingency(table, correction=False)\n    n = table.to_numpy().sum()\n    v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))\n    return {\n        "feature": col,\n        "chi2": float(chi2),\n        "dof": int(dof),\n        "p_value": float(p),\n        "cramers_v": float(v),\n        "n_levels": int(table.shape[0]),\n    }\n\n\ndef rank_auc(x: np.ndarray, y: np.ndarray) -> float:\n    """ROC AUC of score x for binary labels y via the Mann-Whitney U statistic (ties averaged)."""\n    x = np.asarray(x, dtype=float)\n    y = np.asarray(y).astype(bool)\n    n_pos, n_neg = y.sum(), (~y).sum()\n    ranks = ss.rankdata(x)\n    return float((ranks[y].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg))\n\n\ndef bootstrap_auc_ci(\n    x: np.ndarray, y: np.ndarray, n_boot: int = 1000, seed: int = C.SEED, alpha: float = 0.05\n) -> tuple[float, float]:\n    """Stratified bootstrap percentile CI for rank_auc.\n\n    Positives and negatives are resampled separately so every replicate keeps the class counts.\n    """\n    rng = np.random.default_rng(seed)\n    x = np.asarray(x, dtype=float)\n    y = np.asarray(y).astype(bool)\n    pos, neg = x[y], x[~y]\n    lab = np.r_[np.ones(len(pos), bool), np.zeros(len(neg), bool)]\n    aucs = np.empty(n_boot)\n    for b in range(n_boot):\n        xb = np.r_[rng.choice(pos, len(pos)), rng.choice(neg, len(neg))]\n        aucs[b] = rank_auc(xb, lab)\n    return float(np.quantile(aucs, alpha / 2)), float(np.quantile(aucs, 1 - alpha / 2))\n\n\ndef univariate_numeric(\n    df: pd.DataFrame,\n    cols: list[str],\n    target: str = C.TARGET,\n    n_boot: int = 1000,\n    seed: int = C.SEED,\n) -> pd.DataFrame:\n    """Per numeric column: class medians/IQR and univariate ROC AUC with bootstrap CI.\n\n    AUC > 0.5 means larger values are associated with fraud; AUC < 0.5 the opposite.\n    `separation` = |AUC - 0.5| ranks features regardless of direction.\n    """\n    y = df[target].to_numpy()\n    rows = []\n    for i, c in enumerate(cols):\n        x = df[c].to_numpy(dtype=float)\n        auc = rank_auc(x, y)\n        lo, hi = bootstrap_auc_ci(x, y, n_boot=n_boot, seed=seed + i)\n        q = lambda a, p: float(np.quantile(a, p))  # noqa: E731\n        xp, xn = x[y == 1], x[y == 0]\n        rows.append(\n            {\n                "feature": c,\n                "median_fraud": q(xp, 0.5),\n                "iqr_fraud": [q(xp, 0.25), q(xp, 0.75)],\n                "median_legit": q(xn, 0.5),\n                "iqr_legit": [q(xn, 0.25), q(xn, 0.75)],\n                "auc": auc,\n                "auc_ci_low": lo,\n                "auc_ci_high": hi,\n                "separation": abs(auc - 0.5),\n            }\n        )\n    return pd.DataFrame(rows).sort_values("separation", ascending=False).reset_index(drop=True)\n\n\ndef binned_rate(\n    df: pd.DataFrame, col: str, n_bins: int = 10, target: str = C.TARGET\n) -> pd.DataFrame:\n    """Fraud rate by quantile bin of a numeric column, with Wilson CI.\n\n    Low-cardinality integer columns (<= 15 distinct values) are grouped by exact value\n    instead, with the top 1% folded into the 99th-percentile value so no bin is tiny.\n    """\n    if df[col].nunique() <= 15:\n        cap = df[col].quantile(0.99)\n        x = df[col].clip(upper=cap)\n        g = df.groupby(x)[target].agg(n="size", n_fraud="sum").reset_index()\n        g["bin_mid"] = g["bin_left"] = g["bin_right"] = g[col].astype(float)\n    else:\n        bins = pd.qcut(df[col], q=n_bins, duplicates="drop")\n        g = df.groupby(bins, observed=True)[target].agg(n="size", n_fraud="sum").reset_index()\n        g["bin_mid"] = [iv.mid for iv in g[col]]\n        g["bin_left"] = [iv.left for iv in g[col]]\n        g["bin_right"] = [iv.right for iv in g[col]]\n    g["fraud_rate"] = g["n_fraud"] / g["n"]\n    g["ci_low"], g["ci_high"] = wilson_ci(g["n_fraud"].to_numpy(), g["n"].to_numpy())\n    g.insert(0, "feature", col)\n    return g.drop(columns=col)\n', 'src/fraud_e2e/threshold.py': '"""Threshold selection by expected cost on out-of-fold (OOF) predictions.\n\nThe OOF frame has one row per (repeat, transaction) with columns\nrepeat, y, amount_usd, score. Each repeat is a full set of dev predictions, so\neach gives its own cost curve; curves are averaged across repeats and the\nminimum is the chosen threshold (pre-registration A2).\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom fraud_e2e.cost import (\n    CostParams,\n    ThreeTierParams,\n    confusion_by_threshold,\n    cost_from_confusion,\n    require_usable,\n    scaled,\n)\n\nNEAR_OPTIMAL_TOL = 0.01  # thresholds within 1% of the minimum cost are "near-optimal"\n\n\ndef threshold_grid() -> np.ndarray:\n    """Fine grid for 1-D optimisation: dense near 0 (rare-class thresholds), covers [0, 1]."""\n    return np.unique(np.r_[0.0, np.geomspace(1e-4, 1.0, 4001), np.linspace(0, 1, 1001)])\n\n\ndef web_grid(extra=()) -> np.ndarray:\n    """Coarser grid for the website export and the 2-D three-tier search, plus extra points."""\n    return np.unique(np.r_[0.0, np.geomspace(1e-4, 1.0, 481), np.linspace(0, 1, 201), extra])\n\n\ndef near_optimal_range(grid: np.ndarray, mean_curve: np.ndarray, tol=NEAR_OPTIMAL_TOL):\n    """Lowest and highest threshold whose mean cost is within `tol` of the minimum."""\n    ok = mean_curve <= mean_curve.min() * (1 + tol)\n    return float(grid[ok].min()), float(grid[ok].max())\n\n\ndef confusion_per_repeat(oof: pd.DataFrame, grid: np.ndarray) -> dict[int, dict]:\n    return {\n        r: confusion_by_threshold(g["y"], g["score"], g["amount_usd"], grid)\n        for r, g in oof.groupby("repeat")\n    }\n\n\ndef cost_curves(conf: dict[int, dict], p: CostParams) -> np.ndarray:\n    """Array (n_repeats, n_thresholds) of total cost in USD."""\n    return np.vstack([cost_from_confusion(c, p) for c in conf.values()])\n\n\ndef operating_point(conf: dict[int, dict], i: int, n_rows: int) -> dict:\n    """Mean over repeats of the operating-point statistics at grid index i."""\n    tp = np.mean([c["tp"][i] for c in conf.values()])\n    fp = np.mean([c["fp"][i] for c in conf.values()])\n    fn = np.mean([c["fn"][i] for c in conf.values()])\n    return {\n        "recall": tp / (tp + fn),\n        "precision": tp / (tp + fp) if tp + fp > 0 else float("nan"),\n        "alert_rate": (tp + fp) / n_rows,\n        "tp": tp,\n        "fp": fp,\n        "fn": fn,\n    }\n\n\ndef choose_threshold(oof: pd.DataFrame, p: CostParams, grid: np.ndarray | None = None) -> dict:\n    require_usable(p)\n    grid = threshold_grid() if grid is None else grid\n    conf = confusion_per_repeat(oof, grid)\n    curves = cost_curves(conf, p)\n    mean_curve = curves.mean(axis=0)\n    i = int(np.argmin(mean_curve))\n    n_rows = int(oof.groupby("repeat").size().iloc[0])\n    per_repeat_best = [float(grid[np.argmin(c)]) for c in curves]\n    y = oof[oof["repeat"] == oof["repeat"].iloc[0]]\n    nothing = float(\n        (p.fn_loss_fraction * y.loc[y["y"] == 1, "amount_usd"]).sum()\n        + p.fn_fixed_cost * y["y"].sum()\n    )\n    return {\n        "threshold": float(grid[i]),\n        "grid_index": i,\n        "cost_usd": float(mean_curve[i]),\n        "cost_per_1000_txn": float(mean_curve[i] / n_rows * 1000),\n        "cost_sd_across_repeats": float(curves[:, i].std(ddof=1)) if len(curves) > 1 else None,\n        "best_threshold_per_repeat": per_repeat_best,\n        "near_optimal_range": near_optimal_range(grid, mean_curve),\n        "flag_nothing_cost_usd": nothing,\n        "saving_vs_flag_nothing": float(1 - mean_curve[i] / nothing),\n        "n_rows": n_rows,\n        **operating_point(conf, i, n_rows),\n        "_curves": curves,\n        "_grid": grid,\n    }\n\n\ndef sensitivity(oof: pd.DataFrame, base: CostParams, factors=(0.5, 2.0)) -> pd.DataFrame:\n    """Scale one parameter at a time; re-optimise the threshold; also cost of keeping base."""\n    grid = threshold_grid()\n    conf = confusion_per_repeat(oof, grid)\n    n_rows = int(oof.groupby("repeat").size().iloc[0])\n    base_i = int(np.argmin(cost_curves(conf, base).mean(axis=0)))\n    rows = [{"parameter": "(base)", "factor": 1.0, "value": None}]\n    for name in base.numeric():\n        for f in factors:\n            rows.append({"parameter": name, "factor": f, "value": getattr(base, name) * f})\n    out = []\n    for r in rows:\n        p = base if r["parameter"] == "(base)" else scaled(base, r["parameter"], r["factor"])\n        mean_curve = cost_curves(conf, p).mean(axis=0)\n        i = int(np.argmin(mean_curve))\n        lo, hi = near_optimal_range(grid, mean_curve)\n        out.append(\n            {\n                **r,\n                "best_threshold": float(grid[i]),\n                "near_optimal_low": lo,\n                "near_optimal_high": hi,\n                "cost_per_1000_txn": float(mean_curve[i] / n_rows * 1000),\n                "cost_per_1000_at_base_threshold": float(mean_curve[base_i] / n_rows * 1000),\n                "regret_per_1000_txn": float((mean_curve[base_i] - mean_curve[i]) / n_rows * 1000),\n                **{\n                    k: v\n                    for k, v in operating_point(conf, i, n_rows).items()\n                    if k in ("recall", "precision", "alert_rate")\n                },\n            }\n        )\n    return pd.DataFrame(out)\n\n\ndef export_for_web(oof: pd.DataFrame, grid: np.ndarray | None = None) -> dict:\n    """Per-threshold sufficient statistics (mean over repeats) for the static website."""\n    grid = web_grid() if grid is None else grid\n    conf = confusion_per_repeat(oof, grid)\n    n_rows = int(oof.groupby("repeat").size().iloc[0])\n    keys = ["tp", "fp", "fn", "tn", "fn_amount_usd"]\n    mean = {k: np.mean([c[k] for c in conf.values()], axis=0) for k in keys}\n    return {\n        "n_rows": n_rows,\n        "n_fraud": int(mean["tp"][0] + mean["fn"][0]),\n        "fraud_amount_usd": float(\n            oof.loc[oof["y"] == 1].groupby("repeat")["amount_usd"].sum().mean()\n        ),\n        "threshold": grid.round(6).tolist(),\n        **{k: np.round(v, 3).tolist() for k, v in mean.items()},\n        "cost_formula": (\n            "(tp + fp) * review_cost_per_alert + fp * fp_friction_cost"\n            " + fn_loss_fraction * fn_amount_usd + fn * fn_fixed_cost"\n        ),\n        "note": "Mean over 3 CV repeats of out-of-fold predictions on the dev split.",\n    }\n\n\n# --------------------------------------------------------------------------- three-tier\n\n\ndef _prefix_at_grid(y, s, a, grid, p: CostParams):\n    order = np.argsort(s)\n    ys, as_ = y[order].astype(bool), a[order]\n    idx = np.searchsorted(s[order], grid, side="left")  # rows with score < t\n    fn_cost = np.where(ys, p.fn_loss_fraction * as_ + p.fn_fixed_cost, 0.0)\n    pre = lambda v: np.r_[0.0, np.cumsum(v)]  # noqa: E731\n    return {\n        "N": pre(np.ones(len(ys)))[idx],\n        "L": pre(~ys)[idx],\n        "F": pre(ys)[idx],\n        "FNC": pre(fn_cost)[idx],\n        "tot": {\n            "N": len(ys),\n            "L": float((~ys).sum()),\n            "F": float(ys.sum()),\n            "FNC": float(fn_cost.sum()),\n        },\n    }\n\n\ndef three_tier_cost_matrix(y, score, amount, grid, p: CostParams, q: ThreeTierParams):\n    """cost[i, j] for t_stepup = grid[i], t_decline = grid[j] (inf where j < i).\n\n    approve: score < t1; step-up: t1 <= score < t2; decline: score >= t2.\n    """\n    P = _prefix_at_grid(np.asarray(y), np.asarray(score, float), np.asarray(amount, float), grid, p)\n    t = P["tot"]\n    N, L, F, FNC = P["N"], P["L"], P["F"], P["FNC"]\n    approve = FNC[:, None]\n    stepup = (\n        p.review_cost_per_alert * (N[None, :] - N[:, None])\n        + p.fp_friction_cost * (L[None, :] - L[:, None])\n        + q.stepup_fraud_pass_rate * (FNC[None, :] - FNC[:, None])\n    )\n    decline = q.decline_fp_cost * (t["L"] - L[None, :]) + q.decline_tp_cost * (t["F"] - F[None, :])\n    cost = approve + stepup + decline\n    cost[np.tril_indices(len(grid), -1)] = np.inf\n    return cost\n\n\ndef choose_three_tier(\n    oof: pd.DataFrame, p: CostParams, q: ThreeTierParams, grid: np.ndarray | None = None\n) -> dict:\n    require_usable(p, q)\n    grid = web_grid() if grid is None else grid\n    grid = np.r_[grid, np.inf]  # last column = "never decline" (two-tier)\n    mats = [\n        three_tier_cost_matrix(g["y"].to_numpy(), g["score"], g["amount_usd"], grid, p, q)\n        for _, g in oof.groupby("repeat")\n    ]\n    mean = np.mean(mats, axis=0)\n    i, j = np.unravel_index(np.argmin(mean), mean.shape)\n    n_rows = int(oof.groupby("repeat").size().iloc[0])\n    # two-tier under the same three-tier cost model = never decline (t2 beyond max score)\n    two_tier_i = int(np.argmin(mean[:, -1]))\n    return {\n        "t_stepup": float(grid[i]),\n        "t_decline": float(grid[j]),\n        "cost_per_1000_txn": float(mean[i, j] / n_rows * 1000),\n        "two_tier_same_model_threshold": float(grid[two_tier_i]),\n        "two_tier_same_model_cost_per_1000": float(mean[two_tier_i, -1] / n_rows * 1000),\n        "cost_sd_across_repeats_per_1000": float(\n            np.std([m[i, j] for m in mats], ddof=1) / n_rows * 1000\n        ),\n        "best_pair_per_repeat": [\n            [float(grid[a]), float(grid[b])]\n            for a, b in (np.unravel_index(np.argmin(m), m.shape) for m in mats)\n        ],\n        "n_rows": n_rows,\n    }\n', 'configs/cost.yaml': '# Cost assumptions for threshold selection. See notes/01_cost_assumptions.md.\n# ILLUSTRATIVE VALUES — assumed for this portfolio project, NOT sourced from a bank.\nstatus: accepted          # accepted by Nat on 2026-11-09 as illustrative\nlabel: "Illustrative assumed costs (not bank data)"\nparams:\n  review_cost_per_alert: 5.0   # USD per step-up / review (every flagged transaction)\n  fp_friction_cost: 10.0       # USD extra per good customer sent to step-up\n  fn_loss_fraction: 1.0        # multiplier on amount_usd lost on a missed fraud\n  fn_fixed_cost: 15.0          # USD per missed fraud (chargeback handling)\n# Sensitivity analysis: each parameter scaled by these factors, others at base.\nsensitivity_factors: [0.5, 2.0]\n\n# Supplementary three-tier policy (approve / step-up / decline).\n# status must be "accepted" before these are used. See notes/01_cost_assumptions.md.\nthree_tier:\n  status: accepted        # accepted by Nat on 2026-11-09 as illustrative\n  params:\n    stepup_fraud_pass_rate: 0.2   # share of fraud that still gets through step-up\n    decline_fp_cost: 50.0         # USD per good customer hard-declined\n    decline_tp_cost: 0.0          # USD per fraud declined (automatic, no review)\n', 'reports/cv/final.folds.csv': 'repeat,fold,n_train,n_val,n_pos_val,pr_auc,roc_auc,brier,log_loss,base_rate\n0,0,12800,3200,54,0.334216,0.931071,0.0134629,0.0537582,0.016875\n0,1,12800,3200,54,0.395062,0.94219,0.0128172,0.0512681,0.016875\n0,2,12800,3200,54,0.504566,0.929846,0.0112342,0.0479428,0.016875\n0,3,12800,3200,54,0.363856,0.945681,0.01331,0.0526676,0.016875\n0,4,12800,3200,55,0.450863,0.96196,0.0123562,0.046997,0.0171875\n1,0,12800,3200,54,0.44418,0.928339,0.0121084,0.0510991,0.016875\n1,1,12800,3200,54,0.325858,0.936763,0.0134607,0.0536108,0.016875\n1,2,12800,3200,54,0.371267,0.948418,0.0132369,0.051355,0.016875\n1,3,12800,3200,54,0.426737,0.942643,0.012242,0.0490969,0.016875\n1,4,12800,3200,55,0.561682,0.962174,0.0110001,0.0433165,0.0171875\n2,0,12800,3200,54,0.38174,0.94093,0.0128955,0.0513403,0.016875\n2,1,12800,3200,54,0.406909,0.94116,0.0124104,0.0501975,0.016875\n2,2,12800,3200,54,0.415822,0.956294,0.0125344,0.047928,0.016875\n2,3,12800,3200,54,0.442456,0.946257,0.0121722,0.0491173,0.016875\n2,4,12800,3200,55,0.401168,0.923029,0.0129748,0.0542431,0.0171875\n', 'reports/cv/final.spec.json': '{\n  "name": "final",\n  "spec": {\n    "params": {},\n    "imbalance": "none",\n    "drop": [],\n    "calibration": null,\n    "model": "logreg"\n  },\n  "spec_id": "9d9523a408a3"\n}\n', 'reports/p2_modeling.json': '{\n  "_meta": {\n    "data_sha256": "4e32b1ad4880cae452d1f2e5aa53224d38eb913c185455c00a3759df5eb3da61",\n    "seed": 42\n  },\n  "final_pipeline": {\n    "name": "logreg",\n    "spec": {\n      "params": {},\n      "imbalance": "none",\n      "drop": [],\n      "calibration": null,\n      "model": "logreg"\n    },\n    "spec_id": "9d9523a408a3",\n    "pr_auc": {\n      "mean": 0.41509213333333334,\n      "sd": 0.06171861082117158,\n      "ci_low": 0.3406015418051289,\n      "ci_high": 0.48958272486153775\n    },\n    "roc_auc": {\n      "mean": 0.9424503333333332,\n      "sd": 0.011719768346483568,\n      "ci_low": 0.9283052891548013,\n      "ci_high": 0.9565953775118651\n    },\n    "brier": {\n      "mean": 0.012547726666666668,\n      "sd": 0.000741944988783503,\n      "ci_low": 0.01165224440676928,\n      "ci_high": 0.013443208926564056\n    }\n  },\n  "defaults": {\n    "logreg": {\n      "C": 1.0,\n      "max_iter": 2000\n    },\n    "lgbm": {\n      "n_estimators": 300,\n      "learning_rate": 0.05,\n      "num_leaves": 15,\n      "min_child_samples": 20,\n      "subsample": 0.8,\n      "subsample_freq": 1,\n      "colsample_bytree": 0.8,\n      "reg_lambda": 0.0\n    },\n    "xgb": {\n      "n_estimators": 300,\n      "learning_rate": 0.05,\n      "max_depth": 4,\n      "min_child_weight": 0.33228779999999997,\n      "subsample": 0.8,\n      "colsample_bytree": 0.8,\n      "reg_lambda": 1.0\n    }\n  },\n  "decisions": [\n    {\n      "step": "1. boosting library",\n      "a": "lgbm",\n      "b": "xgb",\n      "metric": "pr_auc",\n      "mean_diff_b_minus_a": -0.00392601989830671,\n      "ci_low": -0.034639407952488634,\n      "ci_high": 0.026787368155875214,\n      "p_value": 0.7879631207200878,\n      "n_folds": 15,\n      "b_wins_per_folds": 8,\n      "lower_is_better": false,\n      "verdict": "not distinguishable",\n      "chosen": "lgbm"\n    },\n    {\n      "step": "2. imbalance: class_weight",\n      "a": "lgbm",\n      "b": "lgbm_class_weight",\n      "metric": "pr_auc",\n      "mean_diff_b_minus_a": -0.022525008309437152,\n      "ci_low": -0.058252939510593196,\n      "ci_high": 0.013202922891718896,\n      "p_value": 0.1977514390140954,\n      "n_folds": 15,\n      "b_wins_per_folds": 2,\n      "lower_is_better": false,\n      "verdict": "not distinguishable",\n      "chosen": "lgbm"\n    },\n    {\n      "step": "2. imbalance: undersample",\n      "a": "lgbm",\n      "b": "lgbm_undersample",\n      "metric": "pr_auc",\n      "mean_diff_b_minus_a": -0.004836754794090259,\n      "ci_low": -0.03614160810305788,\n      "ci_high": 0.026468098514877365,\n      "p_value": 0.7452655603968344,\n      "n_folds": 15,\n      "b_wins_per_folds": 5,\n      "lower_is_better": false,\n      "verdict": "not distinguishable",\n      "chosen": "lgbm"\n    },\n    {\n      "step": "2. imbalance: smote",\n      "a": "lgbm",\n      "b": "lgbm_smote",\n      "metric": "pr_auc",\n      "mean_diff_b_minus_a": -0.17736304267555505,\n      "ci_low": -0.22937890022543228,\n      "ci_high": -0.12534718512567783,\n      "p_value": 3.830396571054136e-06,\n      "n_folds": 15,\n      "b_wins_per_folds": 0,\n      "lower_is_better": false,\n      "verdict": "a better",\n      "chosen": "lgbm"\n    },\n    {\n      "step": "3. tuning",\n      "a": "lgbm",\n      "b": "lgbm_tuned",\n      "metric": "pr_auc",\n      "mean_diff_b_minus_a": 0.050146375056174364,\n      "ci_low": 0.014121619593493381,\n      "ci_high": 0.08617113051885535,\n      "p_value": 0.009829213252426111,\n      "n_folds": 15,\n      "b_wins_per_folds": 15,\n      "lower_is_better": false,\n      "verdict": "b better",\n      "chosen": "lgbm_tuned"\n    },\n    {\n      "step": "4. boosting vs logistic",\n      "a": "logreg",\n      "b": "lgbm_tuned",\n      "metric": "pr_auc",\n      "mean_diff_b_minus_a": -0.048384566312167025,\n      "ci_low": -0.08427516358437814,\n      "ci_high": -0.012493969039955914,\n      "p_value": 0.01184123555799934,\n      "n_folds": 15,\n      "b_wins_per_folds": 0,\n      "lower_is_better": false,\n      "verdict": "a better",\n      "chosen": "logreg"\n    },\n    {\n      "a": "logreg_no_ai_flag",\n      "b": "logreg",\n      "metric": "pr_auc",\n      "mean_diff_b_minus_a": 0.02826174790321019,\n      "ci_low": 0.011250608948669121,\n      "ci_high": 0.04527288685775126,\n      "p_value": 0.0031176331488022625,\n      "n_folds": 15,\n      "b_wins_per_folds": 15,\n      "lower_is_better": false,\n      "verdict": "b better",\n      "step": "5. ablation (reported only)"\n    },\n    {\n      "step": "6. calibration: sigmoid",\n      "a": "logreg",\n      "b": "logreg_sigmoid",\n      "metric": "brier",\n      "mean_diff_b_minus_a": -1.2401638546690255e-05,\n      "ci_low": -9.373808071181408e-05,\n      "ci_high": 6.893480361843356e-05,\n      "p_value": 0.748488853519534,\n      "n_folds": 15,\n      "b_wins_per_folds": 5,\n      "lower_is_better": true,\n      "verdict": "not distinguishable",\n      "chosen": "logreg"\n    },\n    {\n      "step": "6. calibration: isotonic",\n      "a": "logreg",\n      "b": "logreg_isotonic",\n      "metric": "brier",\n      "mean_diff_b_minus_a": 5.5662671811585485e-05,\n      "ci_low": -5.211769593090087e-05,\n      "ci_high": 0.00016344303955407184,\n      "p_value": 0.28667462205598665,\n      "n_folds": 15,\n      "b_wins_per_folds": 11,\n      "lower_is_better": true,\n      "verdict": "not distinguishable",\n      "chosen": "logreg"\n    }\n  ],\n  "ablation_ai_flag": {\n    "a": "logreg_no_ai_flag",\n    "b": "logreg",\n    "metric": "pr_auc",\n    "mean_diff_b_minus_a": 0.02826174790321019,\n    "ci_low": 0.011250608948669121,\n    "ci_high": 0.04527288685775126,\n    "p_value": 0.0031176331488022625,\n    "n_folds": 15,\n    "b_wins_per_folds": 15,\n    "lower_is_better": false,\n    "verdict": "b better",\n    "step": "5. ablation (reported only)"\n  },\n  "tuning": {\n    "n_trials": 20,\n    "scored_on": "repeat r0",\n    "best_params": {\n      "num_leaves": 7,\n      "learning_rate": 0.0226,\n      "n_estimators": 800,\n      "min_child_samples": 10,\n      "subsample": 1.0,\n      "colsample_bytree": 0.7,\n      "reg_lambda": 10.0\n    }\n  }\n}\n', 'reports/p2_threshold_web.json': '{\n  "_meta": {\n    "data_sha256": "4e32b1ad4880cae452d1f2e5aa53224d38eb913c185455c00a3759df5eb3da61",\n    "seed": 42\n  },\n  "n_rows": 16000,\n  "n_fraud": 271,\n  "fraud_amount_usd": 46735.69999999999,\n  "threshold": [\n    0.0,\n    0.0001,\n    0.000102,\n    0.000104,\n    0.000106,\n    0.000108,\n    0.00011,\n    0.000112,\n    0.000114,\n    0.000117,\n    0.000119,\n    0.000121,\n    0.000123,\n    0.000126,\n    0.000128,\n    0.000131,\n    0.000133,\n    0.000136,\n    0.000139,\n    0.000141,\n    0.000144,\n    0.000147,\n    0.00015,\n    0.000153,\n    0.000155,\n    0.000158,\n    0.000162,\n    0.000165,\n    0.000168,\n    0.000171,\n    0.000174,\n    0.000178,\n    0.000181,\n    0.000185,\n    0.000188,\n    0.000192,\n    0.000196,\n    0.0002,\n    0.000203,\n    0.000207,\n    0.000211,\n    0.000215,\n    0.00022,\n    0.000224,\n    0.000228,\n    0.000233,\n    0.000237,\n    0.000242,\n    0.000246,\n    0.000251,\n    0.000256,\n    0.000261,\n    0.000266,\n    0.000271,\n    0.000276,\n    0.000282,\n    0.000287,\n    0.000293,\n    0.000299,\n    0.000304,\n    0.00031,\n    0.000316,\n    0.000322,\n    0.000329,\n    0.000335,\n    0.000341,\n    0.000348,\n    0.000355,\n    0.000362,\n    0.000369,\n    0.000376,\n    0.000383,\n    0.000391,\n    0.000398,\n    0.000406,\n    0.000414,\n    0.000422,\n    0.00043,\n    0.000438,\n    0.000447,\n    0.000455,\n    0.000464,\n    0.000473,\n    0.000482,\n    0.000492,\n    0.000501,\n    0.000511,\n    0.000521,\n    0.000531,\n    0.000541,\n    0.000552,\n    0.000562,\n    0.000573,\n    0.000584,\n    0.000596,\n    0.000607,\n    0.000619,\n    0.000631,\n    0.000643,\n    0.000656,\n    0.000668,\n    0.000681,\n    0.000694,\n    0.000708,\n    0.000722,\n    0.000736,\n    0.00075,\n    0.000764,\n    0.000779,\n    0.000794,\n    0.00081,\n    0.000825,\n    0.000841,\n    0.000858,\n    0.000874,\n    0.000891,\n    0.000909,\n    0.000926,\n    0.000944,\n    0.000962,\n    0.000981,\n    0.001,\n    0.001019,\n    0.001039,\n    0.001059,\n    0.00108,\n    0.001101,\n    0.001122,\n    0.001144,\n    0.001166,\n    0.001189,\n    0.001212,\n    0.001235,\n    0.001259,\n    0.001283,\n    0.001308,\n    0.001334,\n    0.001359,\n    0.001386,\n    0.001413,\n    0.00144,\n    0.001468,\n    0.001496,\n    0.001525,\n    0.001555,\n    0.001585,\n    0.001616,\n    0.001647,\n    0.001679,\n    0.001711,\n    0.001744,\n    0.001778,\n    0.001813,\n    0.001848,\n    0.001884,\n    0.00192,\n    0.001957,\n    0.001995,\n    0.002034,\n    0.002073,\n    0.002113,\n    0.002154,\n    0.002196,\n    0.002239,\n    0.002282,\n    0.002326,\n    0.002371,\n    0.002417,\n    0.002464,\n    0.002512,\n    0.002561,\n    0.00261,\n    0.002661,\n    0.002712,\n    0.002765,\n    0.002818,\n    0.002873,\n    0.002929,\n    0.002985,\n    0.003043,\n    0.003102,\n    0.003162,\n    0.003224,\n    0.003286,\n    0.00335,\n    0.003415,\n    0.003481,\n    0.003548,\n    0.003617,\n    0.003687,\n    0.003758,\n    0.003831,\n    0.003905,\n    0.003981,\n    0.004058,\n    0.004137,\n    0.004217,\n    0.004299,\n    0.004382,\n    0.004467,\n    0.004553,\n    0.004642,\n    0.004732,\n    0.004823,\n    0.004917,\n    0.005,\n    0.005012,\n    0.005109,\n    0.005208,\n    0.005309,\n    0.005412,\n    0.005517,\n    0.005623,\n    0.005732,\n    0.005843,\n    0.005957,\n    0.006072,\n    0.00619,\n    0.00631,\n    0.006432,\n    0.006556,\n    0.006683,\n    0.006813,\n    0.006945,\n    0.007079,\n    0.007217,\n    0.007356,\n    0.007499,\n    0.007644,\n    0.007792,\n    0.007943,\n    0.008097,\n    0.008254,\n    0.008414,\n    0.008577,\n    0.008743,\n    0.008913,\n    0.009085,\n    0.009261,\n    0.009441,\n    0.009624,\n    0.00981,\n    0.01,\n    0.010194,\n    0.010391,\n    0.010593,\n    0.010798,\n    0.011007,\n    0.01122,\n    0.011438,\n    0.011659,\n    0.011885,\n    0.012115,\n    0.01235,\n    0.012589,\n    0.012833,\n    0.013082,\n    0.013335,\n    0.013594,\n    0.013857,\n    0.014125,\n    0.014399,\n    0.014678,\n    0.014962,\n    0.015,\n    0.015252,\n    0.015548,\n    0.015849,\n    0.016156,\n    0.016469,\n    0.016788,\n    0.017113,\n    0.017445,\n    0.017783,\n    0.018127,\n    0.018478,\n    0.018836,\n    0.019201,\n    0.019573,\n    0.019953,\n    0.02,\n    0.020339,\n    0.020733,\n    0.021135,\n    0.021544,\n    0.021962,\n    0.022387,\n    0.022821,\n    0.023263,\n    0.023714,\n    0.024173,\n    0.024641,\n    0.025,\n    0.025119,\n    0.025606,\n    0.026102,\n    0.026607,\n    0.027123,\n    0.027648,\n    0.028184,\n    0.02873,\n    0.029286,\n    0.029854,\n    0.03,\n    0.030432,\n    0.031022,\n    0.031623,\n    0.032235,\n    0.03286,\n    0.033497,\n    0.034145,\n    0.034807,\n    0.035,\n    0.035481,\n    0.036169,\n    0.036869,\n    0.037584,\n    0.038312,\n    0.039054,\n    0.039811,\n    0.04,\n    0.040582,\n    0.041368,\n    0.04217,\n    0.042987,\n    0.043819,\n    0.044668,\n    0.045,\n    0.045534,\n    0.046416,\n    0.047315,\n    0.048232,\n    0.049166,\n    0.05,\n    0.050119,\n    0.05109,\n    0.052079,\n    0.053088,\n    0.054117,\n    0.055,\n    0.055165,\n    0.056234,\n    0.057324,\n    0.058434,\n    0.059566,\n    0.06,\n    0.06072,\n    0.061897,\n    0.063096,\n    0.064318,\n    0.065,\n    0.065564,\n    0.066834,\n    0.068129,\n    0.069449,\n    0.069502,\n    0.07,\n    0.070795,\n    0.072166,\n    0.073564,\n    0.074989,\n    0.075,\n    0.076442,\n    0.077923,\n    0.079433,\n    0.08,\n    0.080972,\n    0.08254,\n    0.08414,\n    0.085,\n    0.08577,\n    0.087431,\n    0.089125,\n    0.09,\n    0.090852,\n    0.092612,\n    0.094406,\n    0.095,\n    0.096235,\n    0.098099,\n    0.1,\n    0.101937,\n    0.103912,\n    0.105,\n    0.105925,\n    0.107978,\n    0.11,\n    0.110069,\n    0.112202,\n    0.114376,\n    0.115,\n    0.116591,\n    0.11885,\n    0.12,\n    0.121153,\n    0.1235,\n    0.125,\n    0.125893,\n    0.128332,\n    0.13,\n    0.130818,\n    0.133352,\n    0.135,\n    0.135936,\n    0.138569,\n    0.14,\n    0.141254,\n    0.14399,\n    0.145,\n    0.14678,\n    0.149624,\n    0.15,\n    0.152522,\n    0.155,\n    0.155477,\n    0.158489,\n    0.16,\n    0.16156,\n    0.16469,\n    0.165,\n    0.16788,\n    0.17,\n    0.171133,\n    0.174448,\n    0.175,\n    0.177828,\n    0.18,\n    0.181273,\n    0.184785,\n    0.185,\n    0.188365,\n    0.19,\n    0.192014,\n    0.195,\n    0.195734,\n    0.199526,\n    0.2,\n    0.203392,\n    0.205,\n    0.207332,\n    0.21,\n    0.211349,\n    0.215,\n    0.215443,\n    0.219617,\n    0.22,\n    0.223872,\n    0.225,\n    0.228209,\n    0.23,\n    0.232631,\n    0.235,\n    0.237137,\n    0.24,\n    0.241732,\n    0.245,\n    0.246415,\n    0.25,\n    0.251189,\n    0.255,\n    0.256055,\n    0.26,\n    0.261016,\n    0.265,\n    0.266073,\n    0.27,\n    0.271227,\n    0.275,\n    0.276482,\n    0.28,\n    0.281838,\n    0.285,\n    0.287298,\n    0.29,\n    0.292864,\n    0.295,\n    0.298538,\n    0.3,\n    0.304322,\n    0.305,\n    0.31,\n    0.310218,\n    0.315,\n    0.316228,\n    0.32,\n    0.322354,\n    0.325,\n    0.328599,\n    0.33,\n    0.334965,\n    0.335,\n    0.34,\n    0.341455,\n    0.345,\n    0.34807,\n    0.35,\n    0.354813,\n    0.355,\n    0.36,\n    0.361687,\n    0.365,\n    0.368695,\n    0.37,\n    0.375,\n    0.375837,\n    0.38,\n    0.383119,\n    0.385,\n    0.39,\n    0.390541,\n    0.395,\n    0.398107,\n    0.4,\n    0.405,\n    0.40582,\n    0.41,\n    0.413682,\n    0.415,\n    0.42,\n    0.421697,\n    0.425,\n    0.429866,\n    0.43,\n    0.435,\n    0.438194,\n    0.44,\n    0.445,\n    0.446684,\n    0.45,\n    0.455,\n    0.455337,\n    0.46,\n    0.464159,\n    0.465,\n    0.47,\n    0.473151,\n    0.475,\n    0.48,\n    0.482318,\n    0.485,\n    0.49,\n    0.491662,\n    0.495,\n    0.5,\n    0.501187,\n    0.505,\n    0.51,\n    0.510897,\n    0.515,\n    0.52,\n    0.520795,\n    0.525,\n    0.53,\n    0.530884,\n    0.535,\n    0.54,\n    0.54117,\n    0.545,\n    0.55,\n    0.551654,\n    0.555,\n    0.56,\n    0.562341,\n    0.565,\n    0.57,\n    0.573236,\n    0.575,\n    0.58,\n    0.584341,\n    0.585,\n    0.59,\n    0.595,\n    0.595662,\n    0.6,\n    0.605,\n    0.607202,\n    0.61,\n    0.615,\n    0.618966,\n    0.62,\n    0.625,\n    0.63,\n    0.630957,\n    0.635,\n    0.64,\n    0.643181,\n    0.645,\n    0.65,\n    0.655,\n    0.655642,\n    0.66,\n    0.665,\n    0.668344,\n    0.67,\n    0.675,\n    0.68,\n    0.681292,\n    0.685,\n    0.69,\n    0.694491,\n    0.695,\n    0.7,\n    0.705,\n    0.707946,\n    0.71,\n    0.715,\n    0.72,\n    0.721661,\n    0.725,\n    0.73,\n    0.735,\n    0.735642,\n    0.74,\n    0.745,\n    0.749894,\n    0.75,\n    0.755,\n    0.76,\n    0.764422,\n    0.765,\n    0.77,\n    0.775,\n    0.779232,\n    0.78,\n    0.785,\n    0.79,\n    0.794328,\n    0.795,\n    0.8,\n    0.805,\n    0.809717,\n    0.81,\n    0.815,\n    0.82,\n    0.825,\n    0.825404,\n    0.83,\n    0.835,\n    0.84,\n    0.841395,\n    0.845,\n    0.85,\n    0.855,\n    0.857696,\n    0.86,\n    0.865,\n    0.87,\n    0.874312,\n    0.875,\n    0.88,\n    0.885,\n    0.89,\n    0.891251,\n    0.895,\n    0.9,\n    0.905,\n    0.908518,\n    0.91,\n    0.915,\n    0.92,\n    0.925,\n    0.926119,\n    0.93,\n    0.935,\n    0.94,\n    0.944061,\n    0.945,\n    0.95,\n    0.955,\n    0.96,\n    0.962351,\n    0.965,\n    0.97,\n    0.975,\n    0.98,\n    0.980995,\n    0.985,\n    0.99,\n    0.995,\n    1.0\n  ],\n  "tp": [\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    271.0,\n    270.667,\n    270.667,\n    270.667,\n    270.667,\n    270.667,\n    270.667,\n    270.667,\n    270.667,\n    270.667,\n    270.333,\n    270.333,\n    270.333,\n    270.333,\n    270.333,\n    270.333,\n    270.333,\n    270.333,\n    270.333,\n    270.333,\n    270.0,\n    270.0,\n    270.0,\n    270.0,\n    270.0,\n    270.0,\n    270.0,\n    270.0,\n    269.667,\n    269.667,\n    269.667,\n    269.667,\n    269.667,\n    269.667,\n    269.667,\n    269.667,\n    269.667,\n    269.667,\n    269.667,\n    269.667,\n    269.667,\n    269.667,\n    269.667,\n    269.333,\n    269.333,\n    269.333,\n    269.333,\n    269.0,\n    269.0,\n    269.0,\n    269.0,\n    269.0,\n    269.0,\n    269.0,\n    269.0,\n    269.0,\n    269.0,\n    269.0,\n    269.0,\n    269.0,\n    269.0,\n    269.0,\n    268.667,\n    268.667,\n    268.333,\n    268.333,\n    268.333,\n    268.0,\n    268.0,\n    268.0,\n    268.0,\n    268.0,\n    268.0,\n    268.0,\n    268.0,\n    268.0,\n    268.0,\n    267.667,\n    267.667,\n    267.667,\n    267.667,\n    267.667,\n    267.667,\n    267.667,\n    267.0,\n    267.0,\n    266.667,\n    266.667,\n    266.667,\n    266.333,\n    266.0,\n    266.0,\n    265.667,\n    265.333,\n    265.333,\n    265.333,\n    265.333,\n    265.333,\n    265.333,\n    265.333,\n    265.333,\n    265.333,\n    265.0,\n    265.0,\n    265.0,\n    264.667,\n    264.667,\n    264.667,\n    264.333,\n    264.333,\n    264.333,\n    264.0,\n    264.0,\n    263.667,\n    262.333,\n    262.333,\n    262.333,\n    262.0,\n    262.0,\n    261.667,\n    261.667,\n    261.333,\n    261.333,\n    261.0,\n    261.0,\n    261.0,\n    261.0,\n    261.0,\n    260.333,\n    260.333,\n    260.333,\n    260.333,\n    260.333,\n    260.0,\n    260.0,\n    259.333,\n    259.333,\n    259.0,\n    259.0,\n    258.667,\n    258.667,\n    258.667,\n    258.667,\n    258.667,\n    258.667,\n    258.333,\n    257.0,\n    257.0,\n    256.667,\n    256.667,\n    256.0,\n    255.333,\n    255.0,\n    255.0,\n    254.333,\n    254.333,\n    254.0,\n    254.0,\n    253.333,\n    253.0,\n    253.0,\n    253.0,\n    253.0,\n    252.667,\n    252.667,\n    251.333,\n    250.667,\n    250.667,\n    250.333,\n    250.333,\n    250.333,\n    249.667,\n    249.333,\n    248.667,\n    248.0,\n    247.667,\n    247.333,\n    247.0,\n    246.667,\n    246.0,\n    246.0,\n    246.0,\n    245.333,\n    245.0,\n    244.333,\n    244.0,\n    244.0,\n    244.0,\n    243.667,\n    243.0,\n    242.667,\n    242.667,\n    242.667,\n    242.667,\n    242.0,\n    242.0,\n    241.333,\n    241.333,\n    241.0,\n    241.0,\n    240.667,\n    240.667,\n    240.667,\n    240.667,\n    240.667,\n    240.667,\n    240.333,\n    240.333,\n    240.0,\n    239.0,\n    238.667,\n    238.333,\n    237.333,\n    236.667,\n    236.0,\n    234.667,\n    234.333,\n    233.333,\n    233.333,\n    232.667,\n    231.667,\n    231.667,\n    231.0,\n    230.667,\n    230.333,\n    230.0,\n    229.667,\n    229.667,\n    228.667,\n    228.333,\n    227.667,\n    227.667,\n    227.0,\n    225.0,\n    224.333,\n    224.333,\n    224.333,\n    224.333,\n    223.667,\n    223.0,\n    222.667,\n    222.667,\n    222.333,\n    222.0,\n    221.333,\n    221.0,\n    221.0,\n    220.333,\n    219.0,\n    218.333,\n    217.0,\n    216.667,\n    216.667,\n    215.0,\n    215.0,\n    215.0,\n    215.0,\n    214.667,\n    213.667,\n    213.333,\n    213.0,\n    212.333,\n    212.0,\n    211.667,\n    211.667,\n    211.333,\n    210.667,\n    210.333,\n    210.0,\n    209.667,\n    208.667,\n    208.0,\n    206.333,\n    205.667,\n    205.667,\n    205.333,\n    205.0,\n    205.0,\n    204.0,\n    203.333,\n    202.333,\n    201.333,\n    200.667,\n    200.667,\n    200.333,\n    198.667,\n    198.0,\n    197.333,\n    197.0,\n    195.667,\n    194.667,\n    193.333,\n    192.667,\n    192.333,\n    191.333,\n    190.0,\n    189.333,\n    188.333,\n    188.333,\n    186.333,\n    186.333,\n    185.333,\n    184.667,\n    183.333,\n    183.333,\n    183.333,\n    181.667,\n    181.333,\n    180.667,\n    179.333,\n    178.333,\n    176.667,\n    176.0,\n    176.0,\n    175.333,\n    175.333,\n    175.333,\n    175.0,\n    173.667,\n    173.333,\n    172.333,\n    171.0,\n    169.667,\n    168.667,\n    166.0,\n    163.667,\n    163.0,\n    163.0,\n    162.0,\n    161.667,\n    161.667,\n    160.333,\n    159.333,\n    158.667,\n    158.0,\n    156.667,\n    155.667,\n    155.0,\n    154.667,\n    154.0,\n    153.0,\n    151.333,\n    150.667,\n    150.667,\n    149.333,\n    148.333,\n    148.333,\n    146.0,\n    145.0,\n    144.667,\n    143.667,\n    143.0,\n    142.333,\n    141.333,\n    141.333,\n    140.667,\n    139.333,\n    139.0,\n    138.0,\n    137.667,\n    136.333,\n    135.667,\n    135.0,\n    133.667,\n    133.333,\n    133.0,\n    131.0,\n    130.333,\n    130.333,\n    129.667,\n    129.667,\n    128.0,\n    128.0,\n    126.0,\n    125.333,\n    124.667,\n    124.0,\n    124.0,\n    123.0,\n    123.0,\n    121.0,\n    120.667,\n    120.333,\n    120.333,\n    120.0,\n    117.667,\n    117.667,\n    114.667,\n    114.667,\n    113.0,\n    112.667,\n    111.667,\n    111.0,\n    110.0,\n    109.667,\n    109.0,\n    108.0,\n    108.0,\n    107.333,\n    107.333,\n    107.0,\n    106.667,\n    105.667,\n    105.333,\n    103.0,\n    103.0,\n    101.667,\n    101.0,\n    99.0,\n    99.0,\n    98.0,\n    97.667,\n    96.667,\n    96.333,\n    95.333,\n    94.333,\n    94.333,\n    94.0,\n    93.333,\n    92.0,\n    92.0,\n    90.0,\n    90.0,\n    87.667,\n    87.667,\n    86.333,\n    86.0,\n    86.0,\n    85.0,\n    84.333,\n    83.333,\n    83.0,\n    81.667,\n    81.667,\n    80.333,\n    80.333,\n    80.0,\n    79.333,\n    78.667,\n    78.0,\n    78.0,\n    77.0,\n    77.0,\n    76.667,\n    75.667,\n    75.333,\n    74.333,\n    74.333,\n    74.0,\n    73.333,\n    73.333,\n    71.667,\n    71.333,\n    71.333,\n    70.667,\n    70.667,\n    70.333,\n    70.0,\n    70.0,\n    69.667,\n    69.667,\n    69.667,\n    69.667,\n    69.0,\n    67.0,\n    67.0,\n    66.0,\n    65.0,\n    64.667,\n    63.333,\n    62.667,\n    62.667,\n    62.0,\n    62.0,\n    61.333,\n    60.667,\n    60.667,\n    59.667,\n    59.667,\n    59.667,\n    58.667,\n    58.333,\n    58.0,\n    57.333,\n    57.0,\n    57.0,\n    56.0,\n    56.0,\n    55.333,\n    55.0,\n    54.667,\n    54.667,\n    54.333,\n    54.333,\n    54.333,\n    53.0,\n    53.0,\n    52.667,\n    52.333,\n    52.333,\n    50.667,\n    49.333,\n    49.0,\n    48.667,\n    48.333,\n    48.333,\n    47.333,\n    47.0,\n    46.333,\n    46.333,\n    45.333,\n    45.333,\n    45.333,\n    45.333,\n    45.0,\n    45.0,\n    45.0,\n    45.0,\n    44.333,\n    44.333,\n    42.667,\n    42.333,\n    42.0,\n    41.333,\n    40.667,\n    40.667,\n    40.333,\n    39.333,\n    38.667,\n    38.667,\n    38.0,\n    37.667,\n    37.667,\n    37.333,\n    37.0,\n    37.0,\n    37.0,\n    36.0,\n    34.0,\n    33.667,\n    33.0,\n    32.333,\n    31.667,\n    31.667,\n    30.0,\n    29.0,\n    29.0,\n    29.0,\n    28.667,\n    28.667,\n    28.667,\n    28.333,\n    28.333,\n    27.667,\n    27.667,\n    27.0,\n    25.333,\n    24.333,\n    24.333,\n    23.333,\n    22.333,\n    21.333,\n    21.333,\n    20.333,\n    19.333,\n    19.0,\n    19.0,\n    18.333,\n    17.333,\n    17.333,\n    17.333,\n    17.333,\n    17.333,\n    17.0,\n    17.0,\n    15.0,\n    14.667,\n    14.333,\n    14.333,\n    13.333,\n    13.333,\n    13.333,\n    13.0,\n    11.333,\n    11.333,\n    11.0,\n    10.333,\n    10.333,\n    9.333,\n    9.333,\n    9.333,\n    9.333,\n    8.333,\n    7.667,\n    6.667,\n    6.333,\n    6.333,\n    6.333,\n    6.0,\n    6.0,\n    5.667,\n    5.667,\n    5.667,\n    5.667,\n    5.333,\n    5.333,\n    5.0,\n    4.333,\n    4.0,\n    4.0,\n    3.0,\n    3.0,\n    3.0,\n    2.667,\n    2.333,\n    1.667,\n    0.667,\n    0.333,\n    0.333,\n    0.0,\n    0.0,\n    0.0,\n    0.0\n  ],\n  "fp": [\n    15729.0,\n    14438.667,\n    14412.0,\n    14380.0,\n    14346.667,\n    14313.0,\n    14280.0,\n    14247.0,\n    14219.0,\n    14191.0,\n    14162.0,\n    14126.333,\n    14094.667,\n    14061.667,\n    14028.333,\n    13995.667,\n    13959.667,\n    13923.667,\n    13886.0,\n    13852.667,\n    13813.667,\n    13775.667,\n    13732.333,\n    13696.667,\n    13659.0,\n    13619.333,\n    13580.667,\n    13536.667,\n    13502.333,\n    13465.667,\n    13422.333,\n    13383.667,\n    13349.667,\n    13310.0,\n    13267.667,\n    13232.0,\n    13180.0,\n    13132.667,\n    13085.333,\n    13040.667,\n    12997.333,\n    12954.0,\n    12909.667,\n    12867.0,\n    12828.667,\n    12785.333,\n    12739.0,\n    12686.667,\n    12635.667,\n    12589.0,\n    12539.333,\n    12485.333,\n    12439.333,\n    12388.333,\n    12342.0,\n    12289.667,\n    12239.667,\n    12178.667,\n    12129.667,\n    12076.333,\n    12028.333,\n    11981.667,\n    11933.0,\n    11885.333,\n    11832.667,\n    11790.667,\n    11735.333,\n    11680.667,\n    11628.0,\n    11571.333,\n    11520.333,\n    11455.333,\n    11402.667,\n    11349.0,\n    11301.333,\n    11249.0,\n    11194.0,\n    11139.667,\n    11080.333,\n    11024.333,\n    10971.667,\n    10916.667,\n    10854.0,\n    10791.0,\n    10732.333,\n    10678.0,\n    10614.667,\n    10565.0,\n    10513.0,\n    10459.333,\n    10412.333,\n    10352.333,\n    10302.667,\n    10251.0,\n    10193.0,\n    10134.0,\n    10075.333,\n    10018.333,\n    9968.667,\n    9915.333,\n    9865.667,\n    9810.0,\n    9753.0,\n    9705.0,\n    9651.0,\n    9590.667,\n    9533.0,\n    9480.333,\n    9423.333,\n    9363.0,\n    9299.667,\n    9244.667,\n    9184.667,\n    9118.667,\n    9062.667,\n    9010.667,\n    8941.667,\n    8885.667,\n    8826.0,\n    8761.333,\n    8708.333,\n    8655.333,\n    8604.333,\n    8546.333,\n    8488.333,\n    8431.667,\n    8377.333,\n    8314.333,\n    8256.667,\n    8204.0,\n    8142.333,\n    8079.333,\n    8019.667,\n    7968.0,\n    7900.0,\n    7841.667,\n    7783.333,\n    7731.667,\n    7672.0,\n    7611.333,\n    7558.0,\n    7504.0,\n    7448.667,\n    7390.333,\n    7333.0,\n    7274.0,\n    7221.0,\n    7167.0,\n    7112.0,\n    7059.0,\n    7006.0,\n    6955.333,\n    6905.0,\n    6853.667,\n    6807.0,\n    6743.0,\n    6692.333,\n    6631.0,\n    6573.333,\n    6511.0,\n    6461.667,\n    6413.333,\n    6362.333,\n    6302.333,\n    6251.667,\n    6194.333,\n    6146.333,\n    6098.333,\n    6049.667,\n    5993.0,\n    5942.333,\n    5894.333,\n    5840.667,\n    5788.333,\n    5738.333,\n    5685.333,\n    5633.667,\n    5585.667,\n    5539.667,\n    5494.333,\n    5450.333,\n    5404.333,\n    5351.667,\n    5297.333,\n    5247.0,\n    5200.333,\n    5154.0,\n    5102.0,\n    5052.667,\n    4995.333,\n    4941.667,\n    4895.0,\n    4846.0,\n    4795.667,\n    4749.333,\n    4709.667,\n    4665.333,\n    4615.667,\n    4571.667,\n    4535.0,\n    4491.333,\n    4449.667,\n    4401.0,\n    4357.667,\n    4315.0,\n    4280.667,\n    4273.333,\n    4228.0,\n    4188.0,\n    4147.0,\n    4113.667,\n    4070.0,\n    4024.667,\n    3990.333,\n    3951.333,\n    3907.667,\n    3874.0,\n    3840.667,\n    3801.667,\n    3766.333,\n    3729.667,\n    3690.0,\n    3656.0,\n    3612.667,\n    3572.333,\n    3529.333,\n    3495.333,\n    3458.667,\n    3422.333,\n    3384.667,\n    3348.667,\n    3308.667,\n    3275.667,\n    3238.667,\n    3204.333,\n    3171.333,\n    3136.0,\n    3102.0,\n    3062.333,\n    3026.333,\n    2991.0,\n    2960.0,\n    2927.0,\n    2889.333,\n    2853.333,\n    2814.0,\n    2775.0,\n    2746.667,\n    2715.333,\n    2686.333,\n    2657.0,\n    2623.333,\n    2591.0,\n    2557.0,\n    2524.667,\n    2490.333,\n    2462.667,\n    2430.667,\n    2400.0,\n    2372.333,\n    2348.333,\n    2318.333,\n    2281.333,\n    2258.333,\n    2253.667,\n    2230.333,\n    2200.0,\n    2171.667,\n    2148.0,\n    2125.0,\n    2096.0,\n    2067.333,\n    2042.667,\n    2015.0,\n    1982.0,\n    1957.333,\n    1930.333,\n    1906.333,\n    1877.667,\n    1855.333,\n    1853.0,\n    1832.667,\n    1806.333,\n    1775.333,\n    1754.667,\n    1729.333,\n    1703.333,\n    1682.0,\n    1657.333,\n    1631.0,\n    1606.333,\n    1584.333,\n    1566.667,\n    1559.667,\n    1535.333,\n    1512.333,\n    1488.667,\n    1465.333,\n    1446.0,\n    1425.667,\n    1406.667,\n    1385.333,\n    1362.0,\n    1356.0,\n    1339.0,\n    1319.667,\n    1300.0,\n    1282.0,\n    1259.0,\n    1240.0,\n    1220.667,\n    1205.0,\n    1201.0,\n    1187.667,\n    1167.333,\n    1148.333,\n    1131.333,\n    1113.0,\n    1095.0,\n    1076.0,\n    1072.333,\n    1055.0,\n    1037.0,\n    1019.667,\n    1002.0,\n    986.667,\n    967.0,\n    961.667,\n    950.333,\n    933.667,\n    917.667,\n    901.667,\n    886.667,\n    877.0,\n    875.0,\n    860.0,\n    841.333,\n    820.667,\n    806.333,\n    791.667,\n    789.667,\n    773.667,\n    760.0,\n    741.0,\n    723.333,\n    713.667,\n    705.0,\n    689.667,\n    675.667,\n    662.0,\n    652.333,\n    646.667,\n    633.333,\n    622.667,\n    614.333,\n    614.0,\n    608.667,\n    601.333,\n    587.0,\n    573.333,\n    561.333,\n    561.333,\n    551.0,\n    539.333,\n    528.0,\n    523.0,\n    516.667,\n    505.667,\n    495.333,\n    489.333,\n    485.667,\n    475.0,\n    462.0,\n    460.0,\n    453.0,\n    443.333,\n    435.0,\n    431.333,\n    427.333,\n    416.667,\n    408.667,\n    401.0,\n    392.0,\n    387.0,\n    381.667,\n    373.333,\n    365.333,\n    365.333,\n    356.333,\n    350.333,\n    346.333,\n    341.333,\n    335.667,\n    331.0,\n    328.667,\n    322.333,\n    316.333,\n    312.333,\n    306.667,\n    302.0,\n    300.0,\n    291.667,\n    287.333,\n    284.0,\n    280.0,\n    276.333,\n    274.333,\n    269.0,\n    268.0,\n    266.0,\n    260.333,\n    260.0,\n    258.0,\n    253.667,\n    253.0,\n    246.0,\n    242.0,\n    239.333,\n    233.333,\n    232.667,\n    230.333,\n    224.667,\n    222.0,\n    216.0,\n    215.333,\n    212.667,\n    210.333,\n    208.0,\n    202.333,\n    202.0,\n    197.333,\n    196.333,\n    195.0,\n    193.0,\n    191.333,\n    187.333,\n    187.0,\n    183.333,\n    182.667,\n    178.333,\n    174.667,\n    173.0,\n    168.667,\n    168.667,\n    163.0,\n    162.333,\n    159.0,\n    157.333,\n    154.667,\n    153.0,\n    150.333,\n    149.0,\n    148.0,\n    143.667,\n    142.333,\n    138.333,\n    137.333,\n    135.0,\n    134.0,\n    129.0,\n    128.0,\n    125.333,\n    125.0,\n    124.0,\n    123.0,\n    118.667,\n    117.333,\n    115.667,\n    115.0,\n    112.333,\n    110.333,\n    108.667,\n    107.333,\n    106.0,\n    104.0,\n    101.667,\n    99.667,\n    98.667,\n    96.0,\n    95.667,\n    93.667,\n    93.667,\n    91.0,\n    90.667,\n    88.333,\n    86.333,\n    85.333,\n    83.667,\n    82.667,\n    79.667,\n    79.667,\n    76.667,\n    76.0,\n    75.667,\n    74.667,\n    73.667,\n    70.333,\n    70.333,\n    68.667,\n    68.333,\n    67.0,\n    66.667,\n    66.667,\n    62.667,\n    62.333,\n    59.333,\n    59.0,\n    57.667,\n    55.333,\n    55.333,\n    54.667,\n    53.0,\n    52.667,\n    50.333,\n    50.333,\n    48.333,\n    48.0,\n    47.0,\n    45.333,\n    45.0,\n    43.667,\n    40.667,\n    40.667,\n    39.0,\n    37.333,\n    37.0,\n    36.0,\n    35.667,\n    35.0,\n    33.667,\n    33.667,\n    33.333,\n    32.333,\n    32.333,\n    31.667,\n    31.333,\n    30.333,\n    29.667,\n    29.667,\n    29.0,\n    28.333,\n    28.333,\n    28.333,\n    27.667,\n    27.333,\n    26.0,\n    26.0,\n    26.0,\n    25.667,\n    25.333,\n    25.0,\n    24.333,\n    23.667,\n    23.667,\n    23.333,\n    22.333,\n    22.333,\n    21.667,\n    21.333,\n    20.667,\n    19.333,\n    19.333,\n    19.333,\n    19.333,\n    18.667,\n    18.0,\n    17.667,\n    17.333,\n    16.667,\n    16.333,\n    16.0,\n    16.0,\n    16.0,\n    16.0,\n    16.0,\n    16.0,\n    16.0,\n    14.667,\n    14.333,\n    14.333,\n    13.667,\n    13.667,\n    13.667,\n    13.667,\n    13.333,\n    13.333,\n    13.333,\n    13.0,\n    13.0,\n    13.0,\n    12.333,\n    11.333,\n    11.0,\n    11.0,\n    10.667,\n    9.667,\n    9.333,\n    9.333,\n    9.333,\n    9.0,\n    8.667,\n    8.0,\n    7.667,\n    7.667,\n    7.667,\n    7.333,\n    7.333,\n    7.333,\n    7.0,\n    7.0,\n    6.667,\n    6.333,\n    5.667,\n    5.667,\n    5.667,\n    5.667,\n    5.667,\n    5.333,\n    5.333,\n    5.0,\n    4.333,\n    4.333,\n    4.333,\n    4.333,\n    4.0,\n    3.667,\n    3.0,\n    3.0,\n    2.667,\n    2.667,\n    2.333,\n    2.333,\n    2.333,\n    2.333,\n    2.333,\n    2.333,\n    2.333,\n    2.0,\n    2.0,\n    2.0,\n    2.0,\n    2.0,\n    1.333,\n    1.333,\n    1.333,\n    1.333,\n    1.333,\n    1.0,\n    1.0,\n    1.0,\n    1.0,\n    1.0,\n    1.0,\n    1.0,\n    1.0,\n    1.0,\n    1.0,\n    1.0,\n    1.0,\n    1.0,\n    1.0,\n    1.0,\n    1.0,\n    0.667,\n    0.667,\n    0.667,\n    0.667,\n    0.667,\n    0.333,\n    0.333,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0\n  ],\n  "fn": [\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.333,\n    0.333,\n    0.333,\n    0.333,\n    0.333,\n    0.333,\n    0.333,\n    0.333,\n    0.333,\n    0.667,\n    0.667,\n    0.667,\n    0.667,\n    0.667,\n    0.667,\n    0.667,\n    0.667,\n    0.667,\n    0.667,\n    1.0,\n    1.0,\n    1.0,\n    1.0,\n    1.0,\n    1.0,\n    1.0,\n    1.0,\n    1.333,\n    1.333,\n    1.333,\n    1.333,\n    1.333,\n    1.333,\n    1.333,\n    1.333,\n    1.333,\n    1.333,\n    1.333,\n    1.333,\n    1.333,\n    1.333,\n    1.333,\n    1.667,\n    1.667,\n    1.667,\n    1.667,\n    2.0,\n    2.0,\n    2.0,\n    2.0,\n    2.0,\n    2.0,\n    2.0,\n    2.0,\n    2.0,\n    2.0,\n    2.0,\n    2.0,\n    2.0,\n    2.0,\n    2.0,\n    2.333,\n    2.333,\n    2.667,\n    2.667,\n    2.667,\n    3.0,\n    3.0,\n    3.0,\n    3.0,\n    3.0,\n    3.0,\n    3.0,\n    3.0,\n    3.0,\n    3.0,\n    3.333,\n    3.333,\n    3.333,\n    3.333,\n    3.333,\n    3.333,\n    3.333,\n    4.0,\n    4.0,\n    4.333,\n    4.333,\n    4.333,\n    4.667,\n    5.0,\n    5.0,\n    5.333,\n    5.667,\n    5.667,\n    5.667,\n    5.667,\n    5.667,\n    5.667,\n    5.667,\n    5.667,\n    5.667,\n    6.0,\n    6.0,\n    6.0,\n    6.333,\n    6.333,\n    6.333,\n    6.667,\n    6.667,\n    6.667,\n    7.0,\n    7.0,\n    7.333,\n    8.667,\n    8.667,\n    8.667,\n    9.0,\n    9.0,\n    9.333,\n    9.333,\n    9.667,\n    9.667,\n    10.0,\n    10.0,\n    10.0,\n    10.0,\n    10.0,\n    10.667,\n    10.667,\n    10.667,\n    10.667,\n    10.667,\n    11.0,\n    11.0,\n    11.667,\n    11.667,\n    12.0,\n    12.0,\n    12.333,\n    12.333,\n    12.333,\n    12.333,\n    12.333,\n    12.333,\n    12.667,\n    14.0,\n    14.0,\n    14.333,\n    14.333,\n    15.0,\n    15.667,\n    16.0,\n    16.0,\n    16.667,\n    16.667,\n    17.0,\n    17.0,\n    17.667,\n    18.0,\n    18.0,\n    18.0,\n    18.0,\n    18.333,\n    18.333,\n    19.667,\n    20.333,\n    20.333,\n    20.667,\n    20.667,\n    20.667,\n    21.333,\n    21.667,\n    22.333,\n    23.0,\n    23.333,\n    23.667,\n    24.0,\n    24.333,\n    25.0,\n    25.0,\n    25.0,\n    25.667,\n    26.0,\n    26.667,\n    27.0,\n    27.0,\n    27.0,\n    27.333,\n    28.0,\n    28.333,\n    28.333,\n    28.333,\n    28.333,\n    29.0,\n    29.0,\n    29.667,\n    29.667,\n    30.0,\n    30.0,\n    30.333,\n    30.333,\n    30.333,\n    30.333,\n    30.333,\n    30.333,\n    30.667,\n    30.667,\n    31.0,\n    32.0,\n    32.333,\n    32.667,\n    33.667,\n    34.333,\n    35.0,\n    36.333,\n    36.667,\n    37.667,\n    37.667,\n    38.333,\n    39.333,\n    39.333,\n    40.0,\n    40.333,\n    40.667,\n    41.0,\n    41.333,\n    41.333,\n    42.333,\n    42.667,\n    43.333,\n    43.333,\n    44.0,\n    46.0,\n    46.667,\n    46.667,\n    46.667,\n    46.667,\n    47.333,\n    48.0,\n    48.333,\n    48.333,\n    48.667,\n    49.0,\n    49.667,\n    50.0,\n    50.0,\n    50.667,\n    52.0,\n    52.667,\n    54.0,\n    54.333,\n    54.333,\n    56.0,\n    56.0,\n    56.0,\n    56.0,\n    56.333,\n    57.333,\n    57.667,\n    58.0,\n    58.667,\n    59.0,\n    59.333,\n    59.333,\n    59.667,\n    60.333,\n    60.667,\n    61.0,\n    61.333,\n    62.333,\n    63.0,\n    64.667,\n    65.333,\n    65.333,\n    65.667,\n    66.0,\n    66.0,\n    67.0,\n    67.667,\n    68.667,\n    69.667,\n    70.333,\n    70.333,\n    70.667,\n    72.333,\n    73.0,\n    73.667,\n    74.0,\n    75.333,\n    76.333,\n    77.667,\n    78.333,\n    78.667,\n    79.667,\n    81.0,\n    81.667,\n    82.667,\n    82.667,\n    84.667,\n    84.667,\n    85.667,\n    86.333,\n    87.667,\n    87.667,\n    87.667,\n    89.333,\n    89.667,\n    90.333,\n    91.667,\n    92.667,\n    94.333,\n    95.0,\n    95.0,\n    95.667,\n    95.667,\n    95.667,\n    96.0,\n    97.333,\n    97.667,\n    98.667,\n    100.0,\n    101.333,\n    102.333,\n    105.0,\n    107.333,\n    108.0,\n    108.0,\n    109.0,\n    109.333,\n    109.333,\n    110.667,\n    111.667,\n    112.333,\n    113.0,\n    114.333,\n    115.333,\n    116.0,\n    116.333,\n    117.0,\n    118.0,\n    119.667,\n    120.333,\n    120.333,\n    121.667,\n    122.667,\n    122.667,\n    125.0,\n    126.0,\n    126.333,\n    127.333,\n    128.0,\n    128.667,\n    129.667,\n    129.667,\n    130.333,\n    131.667,\n    132.0,\n    133.0,\n    133.333,\n    134.667,\n    135.333,\n    136.0,\n    137.333,\n    137.667,\n    138.0,\n    140.0,\n    140.667,\n    140.667,\n    141.333,\n    141.333,\n    143.0,\n    143.0,\n    145.0,\n    145.667,\n    146.333,\n    147.0,\n    147.0,\n    148.0,\n    148.0,\n    150.0,\n    150.333,\n    150.667,\n    150.667,\n    151.0,\n    153.333,\n    153.333,\n    156.333,\n    156.333,\n    158.0,\n    158.333,\n    159.333,\n    160.0,\n    161.0,\n    161.333,\n    162.0,\n    163.0,\n    163.0,\n    163.667,\n    163.667,\n    164.0,\n    164.333,\n    165.333,\n    165.667,\n    168.0,\n    168.0,\n    169.333,\n    170.0,\n    172.0,\n    172.0,\n    173.0,\n    173.333,\n    174.333,\n    174.667,\n    175.667,\n    176.667,\n    176.667,\n    177.0,\n    177.667,\n    179.0,\n    179.0,\n    181.0,\n    181.0,\n    183.333,\n    183.333,\n    184.667,\n    185.0,\n    185.0,\n    186.0,\n    186.667,\n    187.667,\n    188.0,\n    189.333,\n    189.333,\n    190.667,\n    190.667,\n    191.0,\n    191.667,\n    192.333,\n    193.0,\n    193.0,\n    194.0,\n    194.0,\n    194.333,\n    195.333,\n    195.667,\n    196.667,\n    196.667,\n    197.0,\n    197.667,\n    197.667,\n    199.333,\n    199.667,\n    199.667,\n    200.333,\n    200.333,\n    200.667,\n    201.0,\n    201.0,\n    201.333,\n    201.333,\n    201.333,\n    201.333,\n    202.0,\n    204.0,\n    204.0,\n    205.0,\n    206.0,\n    206.333,\n    207.667,\n    208.333,\n    208.333,\n    209.0,\n    209.0,\n    209.667,\n    210.333,\n    210.333,\n    211.333,\n    211.333,\n    211.333,\n    212.333,\n    212.667,\n    213.0,\n    213.667,\n    214.0,\n    214.0,\n    215.0,\n    215.0,\n    215.667,\n    216.0,\n    216.333,\n    216.333,\n    216.667,\n    216.667,\n    216.667,\n    218.0,\n    218.0,\n    218.333,\n    218.667,\n    218.667,\n    220.333,\n    221.667,\n    222.0,\n    222.333,\n    222.667,\n    222.667,\n    223.667,\n    224.0,\n    224.667,\n    224.667,\n    225.667,\n    225.667,\n    225.667,\n    225.667,\n    226.0,\n    226.0,\n    226.0,\n    226.0,\n    226.667,\n    226.667,\n    228.333,\n    228.667,\n    229.0,\n    229.667,\n    230.333,\n    230.333,\n    230.667,\n    231.667,\n    232.333,\n    232.333,\n    233.0,\n    233.333,\n    233.333,\n    233.667,\n    234.0,\n    234.0,\n    234.0,\n    235.0,\n    237.0,\n    237.333,\n    238.0,\n    238.667,\n    239.333,\n    239.333,\n    241.0,\n    242.0,\n    242.0,\n    242.0,\n    242.333,\n    242.333,\n    242.333,\n    242.667,\n    242.667,\n    243.333,\n    243.333,\n    244.0,\n    245.667,\n    246.667,\n    246.667,\n    247.667,\n    248.667,\n    249.667,\n    249.667,\n    250.667,\n    251.667,\n    252.0,\n    252.0,\n    252.667,\n    253.667,\n    253.667,\n    253.667,\n    253.667,\n    253.667,\n    254.0,\n    254.0,\n    256.0,\n    256.333,\n    256.667,\n    256.667,\n    257.667,\n    257.667,\n    257.667,\n    258.0,\n    259.667,\n    259.667,\n    260.0,\n    260.667,\n    260.667,\n    261.667,\n    261.667,\n    261.667,\n    261.667,\n    262.667,\n    263.333,\n    264.333,\n    264.667,\n    264.667,\n    264.667,\n    265.0,\n    265.0,\n    265.333,\n    265.333,\n    265.333,\n    265.333,\n    265.667,\n    265.667,\n    266.0,\n    266.667,\n    267.0,\n    267.0,\n    268.0,\n    268.0,\n    268.0,\n    268.333,\n    268.667,\n    269.333,\n    270.333,\n    270.667,\n    270.667,\n    271.0,\n    271.0,\n    271.0,\n    271.0\n  ],\n  "tn": [\n    0.0,\n    1290.333,\n    1317.0,\n    1349.0,\n    1382.333,\n    1416.0,\n    1449.0,\n    1482.0,\n    1510.0,\n    1538.0,\n    1567.0,\n    1602.667,\n    1634.333,\n    1667.333,\n    1700.667,\n    1733.333,\n    1769.333,\n    1805.333,\n    1843.0,\n    1876.333,\n    1915.333,\n    1953.333,\n    1996.667,\n    2032.333,\n    2070.0,\n    2109.667,\n    2148.333,\n    2192.333,\n    2226.667,\n    2263.333,\n    2306.667,\n    2345.333,\n    2379.333,\n    2419.0,\n    2461.333,\n    2497.0,\n    2549.0,\n    2596.333,\n    2643.667,\n    2688.333,\n    2731.667,\n    2775.0,\n    2819.333,\n    2862.0,\n    2900.333,\n    2943.667,\n    2990.0,\n    3042.333,\n    3093.333,\n    3140.0,\n    3189.667,\n    3243.667,\n    3289.667,\n    3340.667,\n    3387.0,\n    3439.333,\n    3489.333,\n    3550.333,\n    3599.333,\n    3652.667,\n    3700.667,\n    3747.333,\n    3796.0,\n    3843.667,\n    3896.333,\n    3938.333,\n    3993.667,\n    4048.333,\n    4101.0,\n    4157.667,\n    4208.667,\n    4273.667,\n    4326.333,\n    4380.0,\n    4427.667,\n    4480.0,\n    4535.0,\n    4589.333,\n    4648.667,\n    4704.667,\n    4757.333,\n    4812.333,\n    4875.0,\n    4938.0,\n    4996.667,\n    5051.0,\n    5114.333,\n    5164.0,\n    5216.0,\n    5269.667,\n    5316.667,\n    5376.667,\n    5426.333,\n    5478.0,\n    5536.0,\n    5595.0,\n    5653.667,\n    5710.667,\n    5760.333,\n    5813.667,\n    5863.333,\n    5919.0,\n    5976.0,\n    6024.0,\n    6078.0,\n    6138.333,\n    6196.0,\n    6248.667,\n    6305.667,\n    6366.0,\n    6429.333,\n    6484.333,\n    6544.333,\n    6610.333,\n    6666.333,\n    6718.333,\n    6787.333,\n    6843.333,\n    6903.0,\n    6967.667,\n    7020.667,\n    7073.667,\n    7124.667,\n    7182.667,\n    7240.667,\n    7297.333,\n    7351.667,\n    7414.667,\n    7472.333,\n    7525.0,\n    7586.667,\n    7649.667,\n    7709.333,\n    7761.0,\n    7829.0,\n    7887.333,\n    7945.667,\n    7997.333,\n    8057.0,\n    8117.667,\n    8171.0,\n    8225.0,\n    8280.333,\n    8338.667,\n    8396.0,\n    8455.0,\n    8508.0,\n    8562.0,\n    8617.0,\n    8670.0,\n    8723.0,\n    8773.667,\n    8824.0,\n    8875.333,\n    8922.0,\n    8986.0,\n    9036.667,\n    9098.0,\n    9155.667,\n    9218.0,\n    9267.333,\n    9315.667,\n    9366.667,\n    9426.667,\n    9477.333,\n    9534.667,\n    9582.667,\n    9630.667,\n    9679.333,\n    9736.0,\n    9786.667,\n    9834.667,\n    9888.333,\n    9940.667,\n    9990.667,\n    10043.667,\n    10095.333,\n    10143.333,\n    10189.333,\n    10234.667,\n    10278.667,\n    10324.667,\n    10377.333,\n    10431.667,\n    10482.0,\n    10528.667,\n    10575.0,\n    10627.0,\n    10676.333,\n    10733.667,\n    10787.333,\n    10834.0,\n    10883.0,\n    10933.333,\n    10979.667,\n    11019.333,\n    11063.667,\n    11113.333,\n    11157.333,\n    11194.0,\n    11237.667,\n    11279.333,\n    11328.0,\n    11371.333,\n    11414.0,\n    11448.333,\n    11455.667,\n    11501.0,\n    11541.0,\n    11582.0,\n    11615.333,\n    11659.0,\n    11704.333,\n    11738.667,\n    11777.667,\n    11821.333,\n    11855.0,\n    11888.333,\n    11927.333,\n    11962.667,\n    11999.333,\n    12039.0,\n    12073.0,\n    12116.333,\n    12156.667,\n    12199.667,\n    12233.667,\n    12270.333,\n    12306.667,\n    12344.333,\n    12380.333,\n    12420.333,\n    12453.333,\n    12490.333,\n    12524.667,\n    12557.667,\n    12593.0,\n    12627.0,\n    12666.667,\n    12702.667,\n    12738.0,\n    12769.0,\n    12802.0,\n    12839.667,\n    12875.667,\n    12915.0,\n    12954.0,\n    12982.333,\n    13013.667,\n    13042.667,\n    13072.0,\n    13105.667,\n    13138.0,\n    13172.0,\n    13204.333,\n    13238.667,\n    13266.333,\n    13298.333,\n    13329.0,\n    13356.667,\n    13380.667,\n    13410.667,\n    13447.667,\n    13470.667,\n    13475.333,\n    13498.667,\n    13529.0,\n    13557.333,\n    13581.0,\n    13604.0,\n    13633.0,\n    13661.667,\n    13686.333,\n    13714.0,\n    13747.0,\n    13771.667,\n    13798.667,\n    13822.667,\n    13851.333,\n    13873.667,\n    13876.0,\n    13896.333,\n    13922.667,\n    13953.667,\n    13974.333,\n    13999.667,\n    14025.667,\n    14047.0,\n    14071.667,\n    14098.0,\n    14122.667,\n    14144.667,\n    14162.333,\n    14169.333,\n    14193.667,\n    14216.667,\n    14240.333,\n    14263.667,\n    14283.0,\n    14303.333,\n    14322.333,\n    14343.667,\n    14367.0,\n    14373.0,\n    14390.0,\n    14409.333,\n    14429.0,\n    14447.0,\n    14470.0,\n    14489.0,\n    14508.333,\n    14524.0,\n    14528.0,\n    14541.333,\n    14561.667,\n    14580.667,\n    14597.667,\n    14616.0,\n    14634.0,\n    14653.0,\n    14656.667,\n    14674.0,\n    14692.0,\n    14709.333,\n    14727.0,\n    14742.333,\n    14762.0,\n    14767.333,\n    14778.667,\n    14795.333,\n    14811.333,\n    14827.333,\n    14842.333,\n    14852.0,\n    14854.0,\n    14869.0,\n    14887.667,\n    14908.333,\n    14922.667,\n    14937.333,\n    14939.333,\n    14955.333,\n    14969.0,\n    14988.0,\n    15005.667,\n    15015.333,\n    15024.0,\n    15039.333,\n    15053.333,\n    15067.0,\n    15076.667,\n    15082.333,\n    15095.667,\n    15106.333,\n    15114.667,\n    15115.0,\n    15120.333,\n    15127.667,\n    15142.0,\n    15155.667,\n    15167.667,\n    15167.667,\n    15178.0,\n    15189.667,\n    15201.0,\n    15206.0,\n    15212.333,\n    15223.333,\n    15233.667,\n    15239.667,\n    15243.333,\n    15254.0,\n    15267.0,\n    15269.0,\n    15276.0,\n    15285.667,\n    15294.0,\n    15297.667,\n    15301.667,\n    15312.333,\n    15320.333,\n    15328.0,\n    15337.0,\n    15342.0,\n    15347.333,\n    15355.667,\n    15363.667,\n    15363.667,\n    15372.667,\n    15378.667,\n    15382.667,\n    15387.667,\n    15393.333,\n    15398.0,\n    15400.333,\n    15406.667,\n    15412.667,\n    15416.667,\n    15422.333,\n    15427.0,\n    15429.0,\n    15437.333,\n    15441.667,\n    15445.0,\n    15449.0,\n    15452.667,\n    15454.667,\n    15460.0,\n    15461.0,\n    15463.0,\n    15468.667,\n    15469.0,\n    15471.0,\n    15475.333,\n    15476.0,\n    15483.0,\n    15487.0,\n    15489.667,\n    15495.667,\n    15496.333,\n    15498.667,\n    15504.333,\n    15507.0,\n    15513.0,\n    15513.667,\n    15516.333,\n    15518.667,\n    15521.0,\n    15526.667,\n    15527.0,\n    15531.667,\n    15532.667,\n    15534.0,\n    15536.0,\n    15537.667,\n    15541.667,\n    15542.0,\n    15545.667,\n    15546.333,\n    15550.667,\n    15554.333,\n    15556.0,\n    15560.333,\n    15560.333,\n    15566.0,\n    15566.667,\n    15570.0,\n    15571.667,\n    15574.333,\n    15576.0,\n    15578.667,\n    15580.0,\n    15581.0,\n    15585.333,\n    15586.667,\n    15590.667,\n    15591.667,\n    15594.0,\n    15595.0,\n    15600.0,\n    15601.0,\n    15603.667,\n    15604.0,\n    15605.0,\n    15606.0,\n    15610.333,\n    15611.667,\n    15613.333,\n    15614.0,\n    15616.667,\n    15618.667,\n    15620.333,\n    15621.667,\n    15623.0,\n    15625.0,\n    15627.333,\n    15629.333,\n    15630.333,\n    15633.0,\n    15633.333,\n    15635.333,\n    15635.333,\n    15638.0,\n    15638.333,\n    15640.667,\n    15642.667,\n    15643.667,\n    15645.333,\n    15646.333,\n    15649.333,\n    15649.333,\n    15652.333,\n    15653.0,\n    15653.333,\n    15654.333,\n    15655.333,\n    15658.667,\n    15658.667,\n    15660.333,\n    15660.667,\n    15662.0,\n    15662.333,\n    15662.333,\n    15666.333,\n    15666.667,\n    15669.667,\n    15670.0,\n    15671.333,\n    15673.667,\n    15673.667,\n    15674.333,\n    15676.0,\n    15676.333,\n    15678.667,\n    15678.667,\n    15680.667,\n    15681.0,\n    15682.0,\n    15683.667,\n    15684.0,\n    15685.333,\n    15688.333,\n    15688.333,\n    15690.0,\n    15691.667,\n    15692.0,\n    15693.0,\n    15693.333,\n    15694.0,\n    15695.333,\n    15695.333,\n    15695.667,\n    15696.667,\n    15696.667,\n    15697.333,\n    15697.667,\n    15698.667,\n    15699.333,\n    15699.333,\n    15700.0,\n    15700.667,\n    15700.667,\n    15700.667,\n    15701.333,\n    15701.667,\n    15703.0,\n    15703.0,\n    15703.0,\n    15703.333,\n    15703.667,\n    15704.0,\n    15704.667,\n    15705.333,\n    15705.333,\n    15705.667,\n    15706.667,\n    15706.667,\n    15707.333,\n    15707.667,\n    15708.333,\n    15709.667,\n    15709.667,\n    15709.667,\n    15709.667,\n    15710.333,\n    15711.0,\n    15711.333,\n    15711.667,\n    15712.333,\n    15712.667,\n    15713.0,\n    15713.0,\n    15713.0,\n    15713.0,\n    15713.0,\n    15713.0,\n    15713.0,\n    15714.333,\n    15714.667,\n    15714.667,\n    15715.333,\n    15715.333,\n    15715.333,\n    15715.333,\n    15715.667,\n    15715.667,\n    15715.667,\n    15716.0,\n    15716.0,\n    15716.0,\n    15716.667,\n    15717.667,\n    15718.0,\n    15718.0,\n    15718.333,\n    15719.333,\n    15719.667,\n    15719.667,\n    15719.667,\n    15720.0,\n    15720.333,\n    15721.0,\n    15721.333,\n    15721.333,\n    15721.333,\n    15721.667,\n    15721.667,\n    15721.667,\n    15722.0,\n    15722.0,\n    15722.333,\n    15722.667,\n    15723.333,\n    15723.333,\n    15723.333,\n    15723.333,\n    15723.333,\n    15723.667,\n    15723.667,\n    15724.0,\n    15724.667,\n    15724.667,\n    15724.667,\n    15724.667,\n    15725.0,\n    15725.333,\n    15726.0,\n    15726.0,\n    15726.333,\n    15726.333,\n    15726.667,\n    15726.667,\n    15726.667,\n    15726.667,\n    15726.667,\n    15726.667,\n    15726.667,\n    15727.0,\n    15727.0,\n    15727.0,\n    15727.0,\n    15727.0,\n    15727.667,\n    15727.667,\n    15727.667,\n    15727.667,\n    15727.667,\n    15728.0,\n    15728.0,\n    15728.0,\n    15728.0,\n    15728.0,\n    15728.0,\n    15728.0,\n    15728.0,\n    15728.0,\n    15728.0,\n    15728.0,\n    15728.0,\n    15728.0,\n    15728.0,\n    15728.0,\n    15728.0,\n    15728.333,\n    15728.333,\n    15728.333,\n    15728.333,\n    15728.333,\n    15728.667,\n    15728.667,\n    15729.0,\n    15729.0,\n    15729.0,\n    15729.0,\n    15729.0,\n    15729.0,\n    15729.0,\n    15729.0,\n    15729.0,\n    15729.0\n  ],\n  "fn_amount_usd": [\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    0.0,\n    7.633,\n    7.633,\n    7.633,\n    7.633,\n    7.633,\n    7.633,\n    7.633,\n    7.633,\n    7.633,\n    15.267,\n    15.267,\n    15.267,\n    15.267,\n    15.267,\n    15.267,\n    15.267,\n    15.267,\n    15.267,\n    15.267,\n    22.9,\n    22.9,\n    22.9,\n    22.9,\n    22.9,\n    22.9,\n    22.9,\n    22.9,\n    211.813,\n    211.813,\n    211.813,\n    211.813,\n    211.813,\n    211.813,\n    211.813,\n    211.813,\n    211.813,\n    211.813,\n    211.813,\n    211.813,\n    211.813,\n    211.813,\n    211.813,\n    400.727,\n    400.727,\n    400.727,\n    400.727,\n    589.64,\n    589.64,\n    589.64,\n    589.64,\n    589.64,\n    589.64,\n    589.64,\n    589.64,\n    589.64,\n    589.64,\n    589.64,\n    589.64,\n    589.64,\n    589.64,\n    589.64,\n    594.65,\n    594.65,\n    602.037,\n    602.037,\n    602.037,\n    607.047,\n    607.047,\n    607.047,\n    607.047,\n    607.047,\n    607.047,\n    607.047,\n    607.047,\n    607.047,\n    607.047,\n    614.433,\n    614.433,\n    614.433,\n    614.433,\n    614.433,\n    614.433,\n    614.433,\n    660.767,\n    660.767,\n    665.777,\n    665.777,\n    665.777,\n    677.117,\n    680.517,\n    680.517,\n    693.133,\n    732.08,\n    732.08,\n    732.08,\n    732.08,\n    732.08,\n    732.08,\n    732.08,\n    732.08,\n    732.08,\n    771.027,\n    771.027,\n    771.027,\n    790.597,\n    790.597,\n    790.597,\n    823.673,\n    823.673,\n    823.673,\n    842.5,\n    842.5,\n    861.327,\n    943.497,\n    943.497,\n    943.497,\n    956.113,\n    956.113,\n    974.94,\n    974.94,\n    986.28,\n    986.28,\n    989.68,\n    989.68,\n    989.68,\n    989.68,\n    989.68,\n    1098.12,\n    1098.12,\n    1098.12,\n    1098.12,\n    1098.12,\n    1117.69,\n    1117.69,\n    1137.167,\n    1137.167,\n    1143.937,\n    1143.937,\n    1149.65,\n    1149.65,\n    1149.65,\n    1149.65,\n    1149.65,\n    1149.65,\n    1203.87,\n    1297.86,\n    1297.86,\n    1606.72,\n    1606.72,\n    1677.277,\n    1746.777,\n    1766.347,\n    1766.347,\n    2091.793,\n    2091.793,\n    2097.507,\n    2097.507,\n    2145.66,\n    2193.79,\n    2193.79,\n    2193.79,\n    2193.79,\n    2254.92,\n    2254.92,\n    2326.79,\n    2350.147,\n    2350.147,\n    2366.597,\n    2366.597,\n    2366.597,\n    2456.243,\n    2464.693,\n    2805.257,\n    2845.41,\n    2857.11,\n    3007.293,\n    3018.993,\n    3080.123,\n    3094.023,\n    3094.023,\n    3094.023,\n    3292.337,\n    3303.067,\n    3340.43,\n    3361.343,\n    3361.343,\n    3361.343,\n    3409.473,\n    3564.133,\n    3592.647,\n    3592.647,\n    3592.647,\n    3592.647,\n    3608.05,\n    3608.05,\n    3953.373,\n    3953.373,\n    3959.193,\n    3959.193,\n    3969.923,\n    3969.923,\n    3969.923,\n    3969.923,\n    3969.923,\n    3969.923,\n    4001.337,\n    4001.337,\n    4003.64,\n    4031.79,\n    4037.61,\n    4069.023,\n    4450.963,\n    4483.587,\n    4514.037,\n    4572.467,\n    4619.443,\n    5017.377,\n    5017.377,\n    5040.103,\n    5431.017,\n    5431.017,\n    5461.077,\n    5479.103,\n    5502.24,\n    5549.217,\n    5555.787,\n    5555.787,\n    6658.043,\n    6676.303,\n    6703.167,\n    6703.167,\n    6726.1,\n    6811.97,\n    7191.357,\n    7191.357,\n    7191.357,\n    7191.357,\n    7295.34,\n    7660.247,\n    7672.28,\n    7672.28,\n    7681.63,\n    7702.06,\n    8250.767,\n    8261.543,\n    8261.543,\n    8325.467,\n    8992.627,\n    9096.61,\n    11258.39,\n    11276.417,\n    11276.417,\n    11959.787,\n    11959.787,\n    11959.787,\n    11959.787,\n    11996.943,\n    12052.8,\n    12172.563,\n    12192.77,\n    12437.113,\n    12438.827,\n    12496.737,\n    12496.737,\n    12513.33,\n    12538.883,\n    12587.493,\n    12598.27,\n    12606.323,\n    12771.013,\n    12905.96,\n    13025.23,\n    13084.207,\n    13084.207,\n    13089.747,\n    13114.467,\n    13114.467,\n    13314.94,\n    13374.293,\n    13571.707,\n    13625.327,\n    14065.12,\n    14065.12,\n    14171.943,\n    14384.25,\n    14521.263,\n    14639.65,\n    14682.427,\n    14726.383,\n    14867.707,\n    14927.51,\n    14972.953,\n    14978.493,\n    15108.427,\n    15125.197,\n    15184.55,\n    15227.537,\n    15227.537,\n    15790.647,\n    15790.647,\n    16279.627,\n    16401.527,\n    16782.633,\n    16782.633,\n    16782.633,\n    17386.093,\n    17402.127,\n    17628.353,\n    17710.16,\n    17836.59,\n    18212.827,\n    18429.28,\n    18429.28,\n    18463.057,\n    18463.057,\n    18463.057,\n    18532.47,\n    18764.08,\n    18773.5,\n    19020.767,\n    19045.303,\n    19113.427,\n    19160.287,\n    19580.583,\n    19704.397,\n    19737.79,\n    19737.79,\n    19805.543,\n    19843.373,\n    19843.373,\n    19872.203,\n    20019.55,\n    20050.377,\n    20080.48,\n    20460.06,\n    20788.097,\n    20870.983,\n    20902.103,\n    20977.263,\n    21015.393,\n    21066.993,\n    21143.65,\n    21143.65,\n    21229.75,\n    21701.133,\n    21701.133,\n    22037.37,\n    22128.9,\n    22206.723,\n    22351.877,\n    22440.397,\n    22727.013,\n    22873.693,\n    22873.693,\n    22893.987,\n    23088.873,\n    23126.547,\n    23158.337,\n    23212.43,\n    23373.327,\n    23385.12,\n    23503.177,\n    23908.57,\n    23985.137,\n    23992.173,\n    24170.977,\n    24391.903,\n    24391.903,\n    24489.05,\n    24489.05,\n    24598.703,\n    24598.703,\n    24728.497,\n    24814.717,\n    24908.363,\n    24958.59,\n    24958.59,\n    25243.99,\n    25243.99,\n    25353.19,\n    25402.987,\n    25422.06,\n    25422.06,\n    25441.133,\n    25954.253,\n    25954.253,\n    26427.533,\n    26427.533,\n    26677.91,\n    26682.52,\n    26999.143,\n    27015.25,\n    27302.73,\n    27428.993,\n    27493.68,\n    27776.283,\n    27776.283,\n    27793.51,\n    27793.51,\n    27843.25,\n    27845.667,\n    27902.07,\n    27948.873,\n    28122.713,\n    28122.713,\n    28345.157,\n    28415.183,\n    28802.067,\n    28802.067,\n    28922.17,\n    28990.333,\n    29047.787,\n    29070.577,\n    29158.88,\n    29238.12,\n    29238.12,\n    29273.417,\n    29497.97,\n    29655.133,\n    29655.133,\n    30077.353,\n    30077.353,\n    30228.85,\n    30228.85,\n    30596.187,\n    30600.153,\n    30600.153,\n    30800.333,\n    30883.05,\n    31288.167,\n    31343.573,\n    31770.75,\n    31770.75,\n    32099.343,\n    32099.343,\n    32143.617,\n    32199.82,\n    32322.283,\n    32423.017,\n    32423.017,\n    32589.117,\n    32589.117,\n    32616.803,\n    32666.92,\n    32690.38,\n    32894.923,\n    32894.923,\n    32930.803,\n    33015.067,\n    33015.067,\n    33267.713,\n    33435.917,\n    33435.917,\n    33481.083,\n    33481.083,\n    33577.32,\n    33661.85,\n    33661.85,\n    33668.99,\n    33668.99,\n    33668.99,\n    33668.99,\n    33759.8,\n    34000.35,\n    34000.35,\n    34732.52,\n    35422.48,\n    35504.78,\n    35589.34,\n    36184.917,\n    36184.917,\n    36685.537,\n    36685.537,\n    36690.323,\n    36728.863,\n    36728.863,\n    36829.19,\n    36829.19,\n    36829.19,\n    37240.053,\n    37326.0,\n    37334.467,\n    37453.91,\n    37459.203,\n    37459.203,\n    37647.837,\n    37647.837,\n    37699.94,\n    37752.87,\n    37775.253,\n    37775.253,\n    37783.72,\n    37783.72,\n    37783.72,\n    37853.147,\n    37853.147,\n    37860.5,\n    37873.42,\n    37873.42,\n    38001.29,\n    38112.02,\n    38117.313,\n    38128.017,\n    38143.65,\n    38143.65,\n    38289.323,\n    38369.143,\n    38454.863,\n    38454.863,\n    38583.347,\n    38583.347,\n    38583.347,\n    38583.347,\n    38636.937,\n    38636.937,\n    38636.937,\n    38636.937,\n    38821.143,\n    38821.143,\n    39092.8,\n    39110.337,\n    39161.22,\n    39371.91,\n    39390.6,\n    39390.6,\n    39426.48,\n    39700.54,\n    39740.147,\n    39740.147,\n    39765.35,\n    39772.54,\n    39772.54,\n    39780.207,\n    39817.653,\n    39817.653,\n    39817.653,\n    39867.203,\n    40151.51,\n    40186.347,\n    40229.87,\n    40352.18,\n    40486.567,\n    40486.567,\n    40628.247,\n    41493.477,\n    41493.477,\n    41493.477,\n    41549.79,\n    41549.79,\n    41549.79,\n    41691.153,\n    41691.153,\n    41898.977,\n    41898.977,\n    42044.76,\n    42312.22,\n    42571.557,\n    42571.557,\n    42616.773,\n    42643.59,\n    42693.16,\n    42693.16,\n    42824.097,\n    43085.797,\n    43101.95,\n    43101.95,\n    43173.183,\n    43322.627,\n    43322.627,\n    43322.627,\n    43322.627,\n    43322.627,\n    44151.067,\n    44151.067,\n    44545.873,\n    44562.683,\n    44569.473,\n    44569.473,\n    44597.15,\n    44597.15,\n    44597.15,\n    44653.04,\n    44812.24,\n    44812.24,\n    44817.37,\n    44865.903,\n    44865.903,\n    45717.36,\n    45717.36,\n    45717.36,\n    45717.36,\n    45749.833,\n    45792.03,\n    45814.74,\n    45818.6,\n    45818.6,\n    45818.6,\n    45825.39,\n    45825.39,\n    45856.18,\n    45856.18,\n    45856.18,\n    45856.18,\n    45886.827,\n    45886.827,\n    45890.687,\n    45946.983,\n    45972.49,\n    45972.49,\n    46059.29,\n    46059.29,\n    46059.29,\n    46091.827,\n    46124.363,\n    46200.89,\n    46542.767,\n    46586.757,\n    46586.757,\n    46735.7,\n    46735.7,\n    46735.7,\n    46735.7\n  ],\n  "cost_formula": "(tp + fp) * review_cost_per_alert + fp * fp_friction_cost + fn_loss_fraction * fn_amount_usd + fn * fn_fixed_cost",\n  "note": "Mean over 3 CV repeats of out-of-fold predictions on the dev split.",\n  "cost_label": "Illustrative assumed costs (not bank data)",\n  "default_params": {\n    "review_cost_per_alert": 5.0,\n    "fp_friction_cost": 10.0,\n    "fn_loss_fraction": 1.0,\n    "fn_fixed_cost": 15.0\n  },\n  "chosen_threshold": 0.0695024317588797,\n  "sensitivity_factors": [\n    0.5,\n    2.0\n  ],\n  "model": {\n    "params": {},\n    "imbalance": "none",\n    "drop": [],\n    "calibration": null,\n    "model": "logreg"\n  },\n  "cv_pr_auc": {\n    "mean": 0.41509213333333334,\n    "sd": 0.06171861082117158,\n    "ci_low": 0.3406015418051289,\n    "ci_high": 0.48958272486153775\n  },\n  "cv_roc_auc": {\n    "mean": 0.9424503333333332,\n    "sd": 0.011719768346483568,\n    "ci_low": 0.9283052891548013,\n    "ci_high": 0.9565953775118651\n  }\n}\n'}
BINARY = {'reports/cv/final.oof.csv.gz': '...'}
for rel, text in FILES.items():
    p = WORK / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(text)
for rel, b64 in BINARY.items():
    p = WORK / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_bytes(base64.b64decode(b64))
sys.path.insert(0, str(WORK / "src"))
from fraud_e2e import config as C
from fraud_e2e.data import load_raw
from fraud_e2e.splits import make_splits, save_splits
if not C.SPLITS_CSV.exists():
    save_splits(make_splits(load_raw()))
print("fraud_e2e ready; data:", C.RAW_CSV)

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from fraud_e2e import config as C
from fraud_e2e import plots as P
from fraud_e2e.cv import load_result
from fraud_e2e.data import load_raw
from fraud_e2e.evaluate import fit_final
from fraud_e2e.explain import global_importance, shap_values
from fraud_e2e.features import feature_frame
from fraud_e2e.reporting import read_json, write_csv, write_json
from fraud_e2e.splits import dev_only

P.set_style()
dev = dev_only(load_raw())
final_spec = read_json("p2_modeling")["final_pipeline"]["spec"]
explain_spec = {**final_spec, "calibration": None}
model = fit_final(dev, explain_spec)
phi, base_value = shap_values(model, dev, explain_spec["drop"])

# sanity check: base + sum of SHAP equals the model's log-odds
raw = model.predict_proba(feature_frame(dev, explain_spec["drop"]))[:, 1]
logit = np.log(raw / (1 - raw))
max_err = float(np.abs(base_value + phi.sum(axis=1).to_numpy() - logit).max())
print(f"explained model: {explain_spec['model']}; additivity check max |error| = {max_err:.2e}")

## 1. Global importance

In [ ]:
imp = global_importance(phi)
write_csv(imp, "p2_shap_global")

fig, ax = P.plt.subplots(figsize=(6.5, 6))
t = imp.iloc[::-1]
ax.barh(t["feature"], t["mean_abs_shap"], color=P.SERIES[0], height=0.6)
ax.set_xlabel("Mean |SHAP| (log-odds), dev rows")
ax.set_title("What drives the fraud score overall")
ax.grid(axis="y", visible=False)
P.save(fig, "p2_shap_global")
P.plt.show()

## 2. Direction of effects

Each dot is one transaction. Horizontal position is that feature's contribution to the score;
colour is the feature's value (low → high, by rank within the feature). For categorical features
colour is not meaningful and is shown grey.

In [ ]:
top = imp["feature"].head(12).tolist()
fig, ax = P.plt.subplots(figsize=(8, 6.5))
rng = np.random.default_rng(C.SEED)
sample = rng.choice(len(dev), size=min(4000, len(dev)), replace=False)
fraud_idx = np.flatnonzero(dev[C.TARGET].to_numpy() == 1)
sample = np.unique(np.r_[sample, fraud_idx])  # keep every fraud visible
cmap = P.plt.get_cmap("coolwarm")
for i, f in enumerate(top[::-1]):
    x = phi[f].to_numpy()[sample]
    yj = i + rng.uniform(-0.3, 0.3, len(sample))
    if f in C.CATEGORICAL_COLS:
        ax.scatter(x, yj, s=4, color=P.NEUTRAL, alpha=0.5, lw=0)
    else:
        v = dev[f].astype(float).to_numpy()[sample]
        col = pd.Series(v).rank(pct=True).to_numpy()
        ax.scatter(x, yj, s=4, c=col, cmap=cmap, vmin=0, vmax=1, alpha=0.6, lw=0)
ax.axvline(0, color=P.NEUTRAL, lw=1)
ax.set_yticks(range(len(top)), top[::-1])
ax.set_xlabel("SHAP value (log-odds contribution to the fraud score)")
ax.grid(axis="y", visible=False)
sm = P.plt.cm.ScalarMappable(cmap=cmap, norm=P.plt.Normalize(0, 1))
fig.colorbar(sm, ax=ax, label="Feature value (rank: 0 = lowest, 1 = highest)")
ax.set_title("Direction and spread of each feature's effect")
P.save(fig, "p2_shap_beeswarm")
P.plt.show()

In [ ]:
num_top = [f for f in top if f in C.NUMERIC_COLS][:3]
fig, axes = P.plt.subplots(1, len(num_top), figsize=(4.4 * len(num_top), 3.4))
units = {
    "merchant_risk_score": "Merchant risk score (0–100)",
    "cvv_retry_count": "CVV retries",
    "velocity_score": "Velocity score (dataset units)",
    "amount_usd": "Amount (USD)",
    "txn_count_last_24h": "Transactions in last 24 h",
    "hours_since_last_txn": "Hours since previous txn",
    "time_of_day_hour": "Hour of day",
    "customer_age": "Customer age (years)",
    "distance_from_home_km": "Distance from home (km)",
    "account_balance_usd": "Balance (USD)",
    "card_age_months": "Card age (months)",
    "prior_disputes": "Prior disputes",
    "day_of_week": "Day of week (0–6)",
}
for ax, f in zip(np.atleast_1d(axes), num_top):
    y = dev[C.TARGET].to_numpy()
    ax.scatter(
        dev[f].to_numpy()[y == 0],
        phi[f].to_numpy()[y == 0],
        s=4,
        alpha=0.3,
        color=P.LEGIT,
        lw=0,
        label="Legit",
    )
    ax.scatter(
        dev[f].to_numpy()[y == 1],
        phi[f].to_numpy()[y == 1],
        s=10,
        alpha=0.8,
        color=P.FRAUD,
        lw=0,
        label="Fraud",
    )
    ax.axhline(0, color=P.NEUTRAL, lw=1)
    ax.set_xlabel(units.get(f, f))
    ax.set_ylabel("SHAP (log-odds)")
    ax.set_title(f)
np.atleast_1d(axes)[0].legend(fontsize=8)
fig.tight_layout()
P.save(fig, "p2_shap_dependence")
P.plt.show()

## 3. Individual transactions

Three dev transactions picked from the final pipeline's **out-of-fold** results at the chosen
step-up threshold (repeat 0), so the outcome category reflects unseen-data behaviour:

- **caught fraud** — the highest-scoring fraud;
- **missed fraud** — the largest-amount fraud below the threshold;
- **false alarm** — the highest-scoring legitimate transaction.

The bars explain the refit (in-sample) model, whose score can differ from the OOF score.

In [ ]:
thr = read_json("p2_threshold_web")["chosen_threshold"]
oof0 = load_result("final").oof.query("repeat == 0").set_index(C.ID_COL)
dev_i = dev.set_index(C.ID_COL)
fraud, legit = oof0[oof0["y"] == 1], oof0[oof0["y"] == 0]
picks = {
    "caught fraud": fraud[fraud["score"] >= thr]["score"].idxmax(),
    "missed fraud": fraud[fraud["score"] < thr]["amount_usd"].idxmax(),
    "false alarm": legit["score"].idxmax(),
}

pos = {tid: i for i, tid in enumerate(dev[C.ID_COL])}
examples = []
fig, axes = P.plt.subplots(1, 3, figsize=(16, 4.8))
for ax, (label, tid) in zip(axes, picks.items()):
    contrib = phi.iloc[pos[tid]].sort_values(key=np.abs, ascending=False)
    shown = contrib.head(8)
    rest = contrib.iloc[8:].sum()
    vals = pd.concat([shown, pd.Series({"(other features)": rest})])[::-1]
    names = [f"{k} = {dev_i.loc[tid, k]}" if k in dev_i.columns else k for k in vals.index]
    ax.barh(names, vals.to_numpy(), color=[P.FRAUD if v > 0 else P.LEGIT for v in vals], height=0.6)
    ax.axvline(0, color=P.NEUTRAL, lw=1)
    logit_i = base_value + contrib.sum()
    p_i = 1 / (1 + np.exp(-logit_i))
    ax.set_title(
        f"{label} (id {tid})\nOOF score {oof0.loc[tid, 'score']:.3f} · "
        f"amount ${dev_i.loc[tid, 'amount_usd']:,.2f}",
        fontsize=10,
    )
    ax.set_xlabel(f"SHAP (log-odds); base {base_value:.2f} → {logit_i:.2f} (p = {p_i:.3f})")
    ax.grid(axis="y", visible=False)
    ax.tick_params(axis="y", labelsize=8)
    examples.append(
        {
            "label": label,
            "transaction_id": int(tid),
            "oof_score": float(oof0.loc[tid, "score"]),
            "amount_usd": float(dev_i.loc[tid, "amount_usd"]),
            "is_fraud": int(dev_i.loc[tid, C.TARGET]),
            "base_log_odds": base_value,
            "log_odds": float(logit_i),
            "contributions": [
                {
                    "feature": k,
                    "value": str(dev_i.loc[tid, k]) if k in dev_i else None,
                    "shap": float(v),
                }
                for k, v in contrib.items()
            ],
        }
    )
fig.suptitle(f"Why these transactions scored as they did (threshold {thr:.4f})", fontweight="bold")
fig.tight_layout()
P.save(fig, "p2_shap_examples")
P.plt.show()

In [ ]:
write_json(
    {
        "explained_model": explain_spec,
        "base_log_odds": base_value,
        "additivity_max_abs_error": max_err,
        "threshold": thr,
        "global_importance": imp.to_dict(orient="records"),
        "examples": examples,
    },
    "p2_shap",
)

## Summary

In [ ]:
top3 = imp.head(3)
share_top3 = top3["mean_abs_shap"].sum() / imp["mean_abs_shap"].sum()
ex = {e["label"]: e for e in examples}


def top_reasons(e, k=3, sign=1):
    c = [x for x in e["contributions"] if np.sign(x["shap"]) == sign][:k]
    return ", ".join(f"`{x['feature']}`" for x in c) or "none"


display(
    Markdown(f"""
- **Top drivers:** {", ".join(f"`{f}`" for f in top3["feature"])} account for
  {100 * share_top3:.0f}% of total mean |SHAP|.
- **Caught fraud** (id {ex["caught fraud"]["transaction_id"]}): pushed up mainly by
  {top_reasons(ex["caught fraud"])}.
- **Missed fraud** (id {ex["missed fraud"]["transaction_id"]}, ${ex["missed fraud"]["amount_usd"]:,.2f}):
  pushed up by {top_reasons(ex["missed fraud"], k=2)} but pulled down by
  {top_reasons(ex["missed fraud"], sign=-1)}; starting from the low base log-odds
  ({base_value:.2f}, fraud is rare) it ended at p = {1 / (1 + np.exp(-ex["missed fraud"]["log_odds"])):.3f},
  below the {thr:.4f} threshold. One strong signal without the other red flags is not enough.
- **False alarm** (id {ex["false alarm"]["transaction_id"]}): pushed up by
  {top_reasons(ex["false alarm"])} — the same red flags as the caught fraud, on a legitimate
  transaction. From these features alone the two are hard to tell apart.
- SHAP values explain the uncalibrated model (additivity error {max_err:.1e}); they describe what
  the model learned from this dataset, not causes of fraud.
""")
)